# 06. 피처 절제 실험 — v2 피처 21개는 값을 하는가 (재튜닝 포함 2×2)

## 이 노트북의 목적

`03_features.ipynb`는 피처를 **두 벌** 만들어 저장해 뒀다.

| 파일 | 피처 수 | 지금 쓰이는 곳 |
|---|---:|---|
| `features_train.parquet` (v1) | 179 | `train.ipynb` / `inference.ipynb` — **현재 확정 모델** |
| `features_v2_train.parquet` (v2) | 200 | 아직 아무 데서도 안 씀 |

v2는 v1에 **21개를 더한 것**이다(v1 컬럼은 값까지 그대로 보존). 문헌을 다시 읽고 도출한
결빙 지속성·구름/안개·복사 안정도·풍향 시어 네 묶음인데, 만들어만 놓고 **효과를 한 번도 재지 않았다.**

이 노트북이 답하는 질문은 하나다: **21개를 켜면 점수가 오르는가, 아니면 노이즈인가.**

## 이 노트북이 왜 이렇게 큰가 — 공정한 비교의 조건 두 가지

이 질문에 정직하게 답하려면 서로 반대 방향으로 기울어진 함정 **두 개**를 동시에 피해야 한다.

### 함정 ① 옛 하이퍼파라미터를 그대로 씌우면 v2가 불리하다

지금 확정 설정(분위수 τ 0.70/0.50/0.65, `T_soft` 0.006, 블렌드 w 0.8/0.5/0.9)은 **v1에 맞춰 고른 값**이다.
피처가 바뀌면 최적 설정도 움직이는 것이 정상인데, 옛 값을 그대로 씌우면 v2는 자기 몸에 안 맞는 옷을 입고 달리는 셈이다.
**피처를 바꿨으면 튜닝도 다시 해야 공정하다.**

### 함정 ② 재튜닝만 하면 이번엔 v2가 유리해진다

그런데 재튜닝만 하고 "v2+새 튜닝 > v1+옛 튜닝"이라고 결론내면, 그 이득이 **피처 덕인지 바뀐 설정 덕인지 구분이 안 된다.**
이 프로젝트는 실제로 그 함정에 빠진 적이 있다 — 피처와 하이퍼파라미터를 함께 바꿔 **+0.0036**이 나왔는데,
바뀐 설정을 *옛 피처에* 그대로 적용해 보니 그중 +0.0024가 설정 덕이었고 **피처 순효과는 +0.0012(노이즈)** 였다.

### 그래서 2×2로 네 칸을 다 채운다

|  | **v1 피처 (179)** | **v2 피처 (200)** |
|---|---|---|
| **v1 기준 튜닝** | **A** — 현 확정 모델 | **B** |
| **v2 기준 튜닝** | **C** — 교란 제거용 팔 | **D** |

여기서 세 가지 양을 뽑는다.

| 양 | 계산 | 뜻 |
|---|---|---|
| 헤드라인 | D − A | "다 바꿨더니 이만큼 올랐다" (유혹적이지만 이걸로 판정하지 않는다) |
| **피처 순효과** | **D − C** | 설정을 똑같이 맞춰 놓고 **피처만** 바꿨을 때의 효과 ← **진짜 판정 축** |
| 설정 효과 | C − A | 새 설정을 옛 피처에 씌웠을 때의 효과 |

C 칸이 없으면 지난번과 똑같은 착각을 반복하게 된다.

> **교란 제거(deconfounding)**: 한 번에 한 가지만 바꿔서 원인을 격리하는 것.
> 약효를 보려면 약만 바꾸고 식단·운동은 그대로 둬야 하는 것과 같다.

## 튜닝은 어디서 하는가 — 2024는 끝까지 건드리지 않는다

**모든 하이퍼파라미터 탐색은 2022~2023 안에서만** 한다. 2024 홀드아웃은 네 칸을 채점할 때 **딱 한 번** 쓴다.

| 탐색 대상 | 탐색 장소 | 근거 |
|---|---|---|
| LightGBM 분위수 τ, 코어 파라미터 | **2022~2023 4개 폴드 CV** | `05_tuning.ipynb` §1·§5와 같은 구조 |
| MLP `T_soft` | **내부검증 = 2023 하반기** | 에폭 수를 정하는 곳과 같은 창 |
| 블렌드 가중치 w | **내부검증 = 2023 하반기** | 두 모델의 내부검증 예측으로 고른다 |

홀드아웃 위에서 설정을 고르면 그 점수는 낙관적으로 부풀려진다(같은 시험지로 공부하고 그 시험지로 채점하는 셈).

## 왜 홀드아웃 하네스를 새로 만드는가

현재 확정 모델의 성능 근거는 **2024 홀드아웃 total 0.6458**이고 `reports/train.md`에 기록돼 있다.
그런데 그 숫자를 **계산하는 코드**는 저장소 어디에도 없다.

- `train.ipynb`는 2022~2024 **전체**로 학습한다(홀드아웃을 남기지 않는다). 최종 산출물을 만드는 코드다.
- `inference.ipynb`는 2025 test를 예측한다. 정답이 없으니 점수를 낼 수 없다.
- `05_tuning.ipynb`는 **LightGBM 단독**만 2024 홀드아웃으로 잰다(0.6308). MLP도 블렌드도 없다.

그래서 §2에서 블렌드까지 포함한 하네스를 만들고, 그것이 기록된 0.6458을 실제로 재현하는지 §3에서 확인한 뒤 진행한다.

> **하네스(harness)**: 부품을 갈아 끼워 가며 같은 트랙에서 랩타임을 재는 시험대.
> 트랙과 계측 방식이 고정돼야 부품 간 비교가 의미를 갖는다.

## 입력·출력·소요 시간

| 구분 | 내용 |
|---|---|
| 입력 | `features_train.parquet`(v1), `features_v2_train.parquet`(v2), `feature_manifest.json` |
| 출력 | 2×2 비교표와 효과 분해, `experiments/log.csv` 추가 행 |
| 건드리지 않는 것 | `train.ipynb` / `inference.ipynb` / `models/final/` — **채택 확정 전에는 확정 모델을 손대지 않는다** |
| 총 소요 | 약 **60~80분** (§3 5분 + §5·§6 튜닝 각 12분 + §7 채점 16분 + §9 절제 20분(조건부)) |

## 누수 점검 (CLAUDE.md 4번)

- 2024는 **채점에만** 쓴다. 학습·조기종료·표준화 통계·하이퍼파라미터 탐색 어디에도 들어가지 않는다.
- v2의 누적 피처(`icing_cum*`, `hours_since_icing`)는 과거 방향 누적이고 재료가 전부 **예보값**이다
  (라벨·SCADA 미참조). `03_features.ipynb` §8-B에서 확인한 내용이다.
- 표준화 통계(mu/sd)는 매번 **그 모델의 학습 구간에서만** 다시 구한다.


## 0. 준비 — 시드 고정과 상수

확정 모델(`train.ipynb`)과 똑같은 설정에서 출발한다. 여기서 바뀌는 것은 피처 목록과 하이퍼파라미터뿐이다.


In [1]:
import os, random, json, sys, time, subprocess, platform, warnings
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import torch

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH
from src.nn import (T_SOFT, MAX_EPOCHS, set_seed, fit_standardizer, standardize,
                    train_mlp, predict_mlp)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RNG_SEED = 42
set_seed(RNG_SEED)
N_THREADS = 4
torch.set_num_threads(N_THREADS)

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
EXP_DIR = REPO_ROOT / "experiments"; EXP_DIR.mkdir(exist_ok=True)

# ---------- 확정 모델(train.ipynb)과 동일한 설정 = 2x2의 'v1 기준 튜닝' ----------
SCORE_THRESHOLD = 0.10
GROUP_TAU  = {"kpx_group_1": 0.70, "kpx_group_2": 0.50, "kpx_group_3": 0.65}
BLEND_W    = {"kpx_group_1": 0.8,  "kpx_group_2": 0.5,  "kpx_group_3": 0.9}
NN_SEEDS   = [42, 1337, 2024, 7, 99]
MAX_ROUNDS, EARLY_STOP = 2000, 50

LGB_PARAMS = dict(objective="quantile", learning_rate=0.05, num_leaves=63, min_data_in_leaf=40,
                  feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbosity=-1, seed=RNG_SEED, num_threads=N_THREADS,
                  deterministic=True, force_row_wise=True)

# ---------- 시간 경계 (train.ipynb보다 1년씩 앞당긴 것) ----------
# train.ipynb: 2022~2024 학습, 내부검증 2024 하반기
# 여기:        2022~2023 학습, 내부검증 2023 하반기, 2024는 채점에만
VAL_START       = pd.Timestamp("2024-01-01 01:00:00")   # 홀드아웃 시작 (05_tuning과 동일)
INNER_VAL_START = pd.Timestamp("2023-07-01 01:00:00")   # 트리 수·에폭 수·T_soft·w 결정용

# ---------- 재튜닝 탐색 범위 ----------
TAU_GRID   = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]       # LightGBM 분위수
N_TRIALS   = 20                                          # Optuna 시도 횟수 (05_tuning과 동일)
TSOFT_GRID = [0.003, 0.006, 0.012, 0.025]                # MLP 계단 근사 폭 (현재값 0.006 포함)
W_GRID     = np.round(np.arange(0.0, 1.001, 0.05), 2)    # 블렌드 가중치

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print(f"Python {platform.python_version()} | LightGBM {lgb.__version__} | PyTorch {torch.__version__}")
print(f"venv 커널 확인: {sys.executable}")
print(f"시드 {RNG_SEED} | 스레드 {N_THREADS} | 현재 T_SOFT {T_SOFT} | 현재 블렌드 w {BLEND_W}")

Python 3.13.14 | LightGBM 4.6.0 | PyTorch 2.13.0+cpu
venv 커널 확인: d:\공모전\wind_forecast\venv\Scripts\python.exe
시드 42 | 스레드 4 | 현재 T_SOFT 0.006 | 현재 블렌드 w {'kpx_group_1': 0.8, 'kpx_group_2': 0.5, 'kpx_group_3': 0.9}


**실행 후 확인할 것**

- `sys.executable` 경로에 `venv`가 들어 있어야 한다(프로젝트 가상환경 커널인지 확인).
- 현재 `T_SOFT`가 **0.006**으로 찍혀야 한다 — `src/nn.py`의 확정값이고, `TSOFT_GRID`에 포함돼 있다.


## 1. 두 피처셋 로딩과 정합성 확인

**이 셀이 하는 일**: v1과 v2를 둘 다 읽고, 비교가 성립하는지 세 가지를 확인한다.

1. **행이 같은가** — 행 수와 시각 순서가 같아야 "같은 데이터, 다른 피처"라고 말할 수 있다.
2. **v2가 v1을 통째로 품고 있는가** — 그래야 차이가 "추가분 21개"로만 설명된다.
3. **v1 컬럼의 값까지 같은가** — 이름만 같고 값이 달라졌다면 추가분의 효과와 뒤섞인다.


In [2]:
feat_v1 = pd.read_parquet(PROCESSED_DIR / "features_train.parquet")
feat_v2 = pd.read_parquet(PROCESSED_DIR / "features_v2_train.parquet")
with open(PROCESSED_DIR / "feature_manifest.json", encoding="utf-8") as fp:
    manifest = json.load(fp)

COLS_V1 = [c for c in feat_v1.columns if c not in TARGET_COLS + ["forecast_kst_dtm"]]
COLS_V2 = [c for c in feat_v2.columns if c not in TARGET_COLS + ["forecast_kst_dtm"]]
V2_ADDED = manifest["v2_added_features"]
V2_BLOCKS = manifest["v2_blocks"]

# (1) 행 정합성
assert len(feat_v1) == len(feat_v2), "v1과 v2의 행 수가 다릅니다"
assert np.array_equal(feat_v1["forecast_kst_dtm"].to_numpy(),
                      feat_v2["forecast_kst_dtm"].to_numpy()), "시각 순서가 다릅니다"

# (2) 상위집합
assert set(COLS_V1).issubset(COLS_V2), "v2가 v1을 품고 있지 않습니다"
assert sorted(set(COLS_V2) - set(COLS_V1)) == sorted(V2_ADDED), "추가 피처 목록이 manifest와 다릅니다"

# (3) v1 컬럼의 값 불변 — 하나라도 바뀌었으면 비교가 교란된다
for c in COLS_V1:
    assert np.array_equal(feat_v1[c].to_numpy(), feat_v2[c].to_numpy()), f"v1 컬럼 {c}의 값이 v2에서 바뀌었습니다"

# 결측·무한대 (모델이 받기 전 마지막 관문)
for nm, f, cols in [("v1", feat_v1, COLS_V1), ("v2", feat_v2, COLS_V2)]:
    assert f[cols].isna().sum().sum() == 0, f"{nm}에 결측이 있습니다"
    assert not np.isinf(f[cols].to_numpy()).any(), f"{nm}에 무한대가 있습니다"

dtm = feat_v1["forecast_kst_dtm"]
holdout_mask = dtm >= VAL_START
HOLDOUT_ANS = feat_v1.loc[holdout_mask, TARGET_COLS].reset_index(drop=True)

print(f"v1 {len(COLS_V1)}개 | v2 {len(COLS_V2)}개 (추가 {len(V2_ADDED)}개)")
print(f"학습 후보 {int((~holdout_mask).sum()):,}행 (~2023) | 홀드아웃 {int(holdout_mask.sum()):,}행 (2024)\n")
print("추가 피처 묶음:")
for k, v in V2_BLOCKS.items():
    print(f"  {k:<22} {len(v):>2}개 : {', '.join(v)}")

v1 179개 | v2 200개 (추가 21개)
학습 후보 17,520행 (~2023) | 홀드아웃 8,784행 (2024)

추가 피처 묶음:
  icing_persistence       6개 : icing_cum6h, icing_cum12h, icing_cum24h, icing_incloud_cum12h, hours_since_icing, melt_potential
  cloud_fog_icing         6개 : lcc, mcc, hcc, vlcdc, icing_flag_lit, icing_incloud
  radiation_stability     6개 : ndnsw, ndnlw, net_radiation, is_daytime, gfs_r850, t850_minus_thub
  veer                    3개 : ldaps_veer_10_50, gfs_veer_10_100, gfs_veer_100_850


**실행 후 확인할 것**

- `v1 179개 | v2 200개 (추가 21개)`
- 홀드아웃이 **8,784행**(2024는 윤년이라 366일 × 24시간)
- assert가 하나도 안 터져야 한다. 터지면 `03_features.ipynb`를 다시 돌려 캐시를 맞춰야 한다.


## 2. 2024 홀드아웃 하네스 — 확정 모델을 1년 앞당겨 재현한다

`train.ipynb`가 하는 일을 **시간만 1년 앞으로 밀어서** 그대로 한다.

| | `train.ipynb` (최종 산출물) | 이 노트북 (성능 측정) |
|---|---|---|
| 학습 기간 | 2022 ~ **2024** 전부 | 2022 ~ **2023** |
| 트리 수·에폭 수 결정 | 2024 하반기 내부검증 | **2023 하반기** 내부검증 |
| 채점 | 없음 (정답을 다 써버렸으므로) | **2024 전체** |

두 모델의 학습 방식은 확정 모델과 글자 하나 다르지 않다.

- **LightGBM 분위수 회귀**: 표본가중 = 실제 발전량(`actual`), 라벨 있는 행 전부로 학습.
  트리 수는 내부검증으로 정한 뒤 **데이터가 늘어난 비율만큼 곱해서** 늘린다
  (조기 종료로 정한 개수는 그 데이터 양에 맞는 값이라, 데이터가 늘면 그만큼 더 필요하다).
- **MLP + 산식 손실**: 채점 대상 행만, 시드 5개 평균. 에폭 수는 내부검증 점수가 가장 높았던 시점의 **중앙값**.

**하이퍼파라미터는 이제 인자로 받는다.** 2×2의 네 칸이 같은 코드를 쓰되 설정만 갈아 끼우게 하기 위해서다.
설정 묶음(`hp`)의 형태는 아래와 같다.

```
hp = {"lgb":    {그룹: LightGBM 파라미터 dict},
      "t_soft": MLP 계단 근사 폭,
      "w":      {그룹: 블렌드에서 MLP가 차지하는 비중}}
```


In [3]:
def build_masks(feat, group):
    """한 그룹의 학습/내부검증 마스크. scored=채점 대상(실제 이용률 >= 10%)."""
    lab = feat[group].notna()
    sc = feat[group] >= CAPACITY_KWH[group] * SCORE_THRESHOLD
    before_val = feat["forecast_kst_dtm"] < VAL_START          # 2024를 절대 넣지 않는다
    before_inner = feat["forecast_kst_dtm"] < INNER_VAL_START
    return {
        "fit_all":         lab & before_val,
        "fit_scored":      lab & sc & before_val,
        "inner_tr":        lab & before_inner,
        "inner_tr_scored": lab & sc & before_inner,
        "inner_va":        lab & (~before_inner) & before_val,
        "inner_va_scored": lab & sc & (~before_inner) & before_val,
    }


def group_score(actual, forecast, capacity):
    """metric()과 같은 산식을 한 그룹만 계산하도록 뗀 것.
    src/metric.py는 대회 공식 산식이라 수정하지 않는다는 규칙(CLAUDE.md 4번) 때문에 여기서 재정의한다."""
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    valid = actual >= capacity * 0.10
    if not np.any(valid):
        return float("nan"), float("nan"), float("nan")
    actual, forecast = actual[valid], forecast[valid]
    error_rate = np.abs(forecast - actual) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = np.sum(actual * unit_price) / np.sum(actual * 4.0)
    return 0.5 * (1.0 - nmae) + 0.5 * ficr, nmae, ficr


def train_lgb_holdout(feat, cols, group, params):
    """2022~2023으로 학습해 2024를 예측한다. train.ipynb의 train_lgb와 같은 절차."""
    m = build_masks(feat, group)

    d_tr = lgb.Dataset(feat.loc[m["inner_tr"], cols], label=feat.loc[m["inner_tr"], group],
                       weight=feat.loc[m["inner_tr"], group].to_numpy())
    d_va = lgb.Dataset(feat.loc[m["inner_va_scored"], cols],
                       label=feat.loc[m["inner_va_scored"], group], reference=d_tr)
    probe = lgb.train(params, d_tr, MAX_ROUNDS, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])

    ratio = m["fit_all"].sum() / m["inner_tr"].sum()
    n_final = max(int(round(probe.best_iteration * ratio)), 50)

    d_full = lgb.Dataset(feat.loc[m["fit_all"], cols], label=feat.loc[m["fit_all"], group],
                         weight=feat.loc[m["fit_all"], group].to_numpy())
    booster = lgb.train(params, d_full, n_final)

    pred = np.clip(booster.predict(feat.loc[holdout_mask, cols]), 0, CAPACITY_KWH[group])
    return pred, {"best_iter_inner": int(probe.best_iteration), "n_trees": int(booster.num_trees())}


def train_mlp_holdout(feat, cols, group, t_soft):
    """2022~2023으로 시드 5개를 학습해 2024를 예측한다. train.ipynb의 train_mlp_group과 같은 절차.
    pred_seeds(시드별 예측)를 함께 돌려준다 -- §8에서 '시드 노이즈'를 재는 데 쓴다."""
    cap = CAPACITY_KWH[group]
    m = build_masks(feat, group)

    # --- 1단계: 내부검증(2023 하반기)으로 에폭 수 결정 ---
    X_itr = feat.loc[m["inner_tr_scored"], cols].to_numpy(np.float32)
    y_itr = (feat.loc[m["inner_tr_scored"], group] / cap).to_numpy(np.float32)
    X_iva = feat.loc[m["inner_va_scored"], cols].to_numpy(np.float32)
    y_iva = (feat.loc[m["inner_va_scored"], group] / cap).to_numpy(np.float32)

    mu_i, sd_i = fit_standardizer(X_itr)                  # 내부학습에서만 fit
    Xi, Xv = (X_itr - mu_i) / sd_i, standardize(X_iva, mu_i, sd_i)

    def eval_fn(model):
        with torch.no_grad():
            p = np.clip(model(Xv).numpy(), 0, 1)
        return group_score(y_iva * cap, p * cap, cap)[0]

    epochs = [train_mlp(Xi, y_itr, seed=s, n_epochs=MAX_EPOCHS, t_soft=t_soft, eval_fn=eval_fn)[1]
              for s in NN_SEEDS]
    ep_final = int(np.median(epochs))

    # --- 2단계: 2022~2023 전체로 재학습 (조기 종료 없이 ep_final 에폭) ---
    X_full = feat.loc[m["fit_scored"], cols].to_numpy(np.float32)
    y_full = (feat.loc[m["fit_scored"], group] / cap).to_numpy(np.float32)
    mu, sd = fit_standardizer(X_full)                     # 최종 학습 데이터에서만 fit (2024 미포함)
    Xf = (X_full - mu) / sd

    X_ho = feat.loc[holdout_mask, cols].to_numpy(np.float32)
    pred_seeds = []
    for s in NN_SEEDS:
        model = train_mlp(Xf, y_full, seed=s, n_epochs=ep_final, t_soft=t_soft, eval_fn=None)[0]
        pred_seeds.append(predict_mlp(model, X_ho, mu, sd, cap))
    pred_seeds = np.array(pred_seeds)                     # (시드, 홀드아웃행)

    return pred_seeds.mean(axis=0), pred_seeds, {"epochs_inner": epochs, "n_epochs": ep_final,
                                                 "n_train_rows": int(m["fit_scored"].sum())}


def run_config(feat, cols, hp, name, verbose=True):
    """한 (피처셋, 하이퍼파라미터) 조합으로 학습하고 2024 홀드아웃 점수를 낸다."""
    t0 = time.time()
    p_lgb, p_mlp, p_blend, p_seeds, info = {}, {}, {}, {}, {}

    for g in TARGET_COLS:
        cap = CAPACITY_KWH[g]
        p_lgb[g], i_lgb = train_lgb_holdout(feat, cols, g, hp["lgb"][g])
        p_mlp[g], p_seeds[g], i_mlp = train_mlp_holdout(feat, cols, g, hp["t_soft"])
        w = hp["w"][g]
        p_blend[g] = np.clip((1 - w) * p_lgb[g] + w * p_mlp[g], 0, cap)
        info[g] = {**i_lgb, **i_mlp}
        if verbose:
            print(f"  {g}: 트리 {i_lgb['n_trees']}그루 | MLP {i_mlp['n_epochs']}에폭 x 시드 {len(NN_SEEDS)}개 | w={w}")

    out = {"name": name, "n_features": len(cols), "hp": hp, "info": info,
           "pred": {"lgb": p_lgb, "mlp": p_mlp, "blend": p_blend}, "pred_seeds": p_seeds,
           "elapsed": time.time() - t0}

    for key, pr in [("lgb", p_lgb), ("mlp", p_mlp), ("blend", p_blend)]:
        t, n, f = metric(HOLDOUT_ANS, pd.DataFrame(pr))
        out[key] = {"total": t, "one_minus_nmae": n, "ficr": f}
    out["per_group"] = {g: group_score(HOLDOUT_ANS[g], p_blend[g], CAPACITY_KWH[g])[0] for g in TARGET_COLS}
    return out


# 2x2의 'v1 기준 튜닝' = 현재 확정 모델의 설정
HP_V1 = {"lgb": {g: {**LGB_PARAMS, "alpha": GROUP_TAU[g]} for g in TARGET_COLS},
         "t_soft": T_SOFT, "w": dict(BLEND_W)}

print("하네스 준비 완료 — 한 조합 채점에 4~5분 (LightGBM 3개 + MLP 30개 학습)")
for g in TARGET_COLS:
    m = build_masks(feat_v1, g)
    print(f"  {g}: 최종학습 {int(m['fit_all'].sum()):,}행 (채점행 {int(m['fit_scored'].sum()):,}) "
          f"| 내부검증 {int(m['inner_va'].sum()):,}행 | 홀드아웃 채점행 "
          f"{int((HOLDOUT_ANS[g] >= CAPACITY_KWH[g] * SCORE_THRESHOLD).sum()):,}")

하네스 준비 완료 — 한 조합 채점에 4~5분 (LightGBM 3개 + MLP 30개 학습)
  kpx_group_1: 최종학습 17,422행 (채점행 10,925) | 내부검증 4,413행 | 홀드아웃 채점행 4,990
  kpx_group_2: 최종학습 17,423행 (채점행 10,914) | 내부검증 4,414행 | 홀드아웃 채점행 4,977
  kpx_group_3: 최종학습 8,760행 (채점행 4,847) | 내부검증 4,416행 | 홀드아웃 채점행 4,567


**실행 후 확인할 것**

- 이 셀은 함수 정의뿐이라 몇 초면 끝난다. 학습은 다음 셀부터다.
- 그룹 3의 최종학습 행 수가 나머지 둘보다 **눈에 띄게 적어야** 정상이다(라벨이 2023년부터만 있다).


## 3. 칸 A — v1 피처 + v1 튜닝 (현 확정 모델). 0.6458이 다시 나오는가

**측정 도구부터 검증한다.** 2×2를 채우기 전에, 이 하네스가 기록된 확정 모델 성능을 재현하는지 본다.

| 기록된 값 (`reports/train.md`) | total | 1−NMAE | FICR |
|---|---:|---:|---:|
| LightGBM 단독 | 0.6308 | 0.8636 | 0.3979 |
| MLP 단독 | 0.6435 | 0.8736 | 0.4134 |
| **블렌드** | **0.6458** | **0.8744** | **0.4172** |

> **LightGBM 부분은 노트북을 짜면서 미리 확인했다.** 이 하네스의 LightGBM 경로만 따로 돌려 보니
> `total 0.630751 / 1-NMAE 0.863555 / FICR 0.397948` — `experiments/log.csv`의 exp011과
> **소수점 여섯 자리까지 완전히 같다.** 마스크·파라미터·트리 수 산정이 05_tuning과 동일하다는 뜻이다.
> 따라서 아래에서 LightGBM 줄은 **정확히 0.630751**이 나와야 하고, 확인이 필요한 것은 MLP와 블렌드 쪽이다.

**여기서 크게 어긋나면 아래로 진행하면 안 된다.** 저울이 틀렸는데 무게를 재는 셈이 된다.

⚠️ 참고로 확정 모델의 w(0.8/0.5/0.9)는 과거에 **이 홀드아웃 위에서** 고른 값이라 A는 약간 낙관적이다.
§5·§6에서 새로 고르는 w는 내부검증에서만 고르므로 그 이점이 없다. 이 비대칭은 **A에게 유리**한 쪽이니,
v2가 그럼에도 A를 이긴다면 그 결론은 보수적으로 안전하다.

⏱️ **4~5분 소요**


In [4]:
print("=== 칸 A: v1 피처(179) + v1 튜닝 (현 확정 모델) ===")
res_A = run_config(feat_v1, COLS_V1, HP_V1, "A_v1feat_v1hp")

print(f"\n{'':<12}{'total':>9}{'1-NMAE':>9}{'FICR':>9}   (기록값 대비)")
REF = {"lgb": 0.6308, "mlp": 0.6435, "blend": 0.6458}
for key, label in [("lgb", "LightGBM"), ("mlp", "MLP"), ("blend", "블렌드")]:
    r = res_A[key]
    print(f"{label:<12}{r['total']:>9.4f}{r['one_minus_nmae']:>9.4f}{r['ficr']:>9.4f}   {r['total'] - REF[key]:+.4f}")

print("\n그룹별 블렌드 점수:")
for g, s in res_A["per_group"].items():
    print(f"  {g}: {s:.4f}")
print(f"\n소요 {res_A['elapsed']:.0f}초")

=== 칸 A: v1 피처(179) + v1 튜닝 (현 확정 모델) ===
  kpx_group_1: 트리 187그루 | MLP 31에폭 x 시드 5개 | w=0.8
  kpx_group_2: 트리 304그루 | MLP 76에폭 x 시드 5개 | w=0.5
  kpx_group_3: 트리 177그루 | MLP 31에폭 x 시드 5개 | w=0.9

                total   1-NMAE     FICR   (기록값 대비)
LightGBM       0.6308   0.8636   0.3979   -0.0000
MLP            0.6435   0.8736   0.4133   -0.0000
블렌드            0.6458   0.8744   0.4172   +0.0000

그룹별 블렌드 점수:
  kpx_group_1: 0.6512
  kpx_group_2: 0.6764
  kpx_group_3: 0.6099

소요 135초


**실행 후 확인할 것**

- **LightGBM total이 정확히 0.630751** — 여기가 어긋나면 마스크나 파라미터가 다른 것이다.
- **블렌드 total이 0.6458 근처(±0.003)** — MLP는 하드웨어에 따라 부동소수점 연산 순서가 달라 조금 흔들릴 수 있다.
- 세 줄이 `블렌드 > MLP > LightGBM` 순서 — 블렌드가 두 모델보다 나은 것이 이 구조의 존재 이유다.

이 조건이 깨지면 아래로 진행하지 말고 원인을 먼저 찾는다.


## 4. 재튜닝 도구 — 2022~2023 안에서만 설정을 고른다

세 종류의 하이퍼파라미터를 각각 다른 방식으로 고른다. **어느 것도 2024를 보지 않는다.**

### 4-1. LightGBM — 2022~2023 4개 폴드 교차검증

`05_tuning.ipynb` §1과 **똑같은 폴드**를 쓴다. 계절이 한 바퀴 돌도록 3개월씩 네 조각으로 나누고,
각 폴드는 검증 직전까지의 모든 데이터로 학습한다(확장 윈도 — 시간을 거스르지 않는다).

| 폴드 | 검증 구간 |
|---|---|
| F1 겨울 | 2023-01 ~ 03 |
| F2 봄 | 2023-04 ~ 06 |
| F3 여름 | 2023-07 ~ 09 |
| F4 가을 | 2023-10 ~ 12 |

그룹 3은 라벨이 2023년부터라 F1은 건너뛴다(학습 데이터가 없다).

두 가지를 탐색하고 **CV 점수가 높은 쪽을 그룹마다 채택**한다.

1. **τ 격자 탐색** — 분위수만 훑는다(코어 파라미터는 기본값 고정)
2. **Optuna 20회** — τ + 학습률·잎 개수·최소 잎 샘플·피처/배깅 비율·정규화까지 함께 탐색

> **분위수 회귀의 τ**: "내 예측보다 실제가 낮을 확률이 τ가 되도록" 맞추는 것.
> τ=0.5면 중앙값, τ가 크면 예측을 위로 밀어 올린다. FICR이 발전량 가중이라 위로 미는 편이 유리해서 τ>0.5가 나온다.

> **Optuna**: 이전 시도 결과를 보고 다음에 시험할 설정을 똑똑하게 고르는 자동 탐색 도구.
> 격자를 전부 훑는 것보다 훨씬 적은 횟수로 좋은 값을 찾는다.

### 4-2. MLP `T_soft` — 내부검증(2023 하반기)

`T_soft`는 계단 함수인 FICR 단가를 얼마나 부드럽게 뭉갤지 정하는 폭이다.
작으면 실제 계단에 가깝지만 기울기가 날카로워 학습이 불안정해지고, 크면 학습은 안정되나 산식과 멀어진다.

에폭 수를 정하는 것과 **같은 창(2023 하반기)** 에서 고른다. 후보마다 MLP를 전부 새로 학습해야 해서 이 노트북에서 가장 비싼 부분이다.

### 4-3. 블렌드 가중치 w — 내부검증(2023 하반기)

4-2를 돌리면서 **내부검증 구간의 예측**을 함께 받아 둔다. 거기에 LightGBM의 내부검증 예측을 겹쳐
그룹마다 점수가 가장 높은 w를 0.05 간격으로 고른다. 이미 만든 예측을 재사용하므로 **추가 학습이 없다**(거의 공짜).


In [5]:
# ---------- 폴드 정의 (05_tuning.ipynb §1과 동일) ----------
FOLDS = [
    ("F1 겨울", pd.Timestamp("2023-01-01 01:00:00"), pd.Timestamp("2023-04-01 00:00:00")),
    ("F2 봄",   pd.Timestamp("2023-04-01 01:00:00"), pd.Timestamp("2023-07-01 00:00:00")),
    ("F3 여름", pd.Timestamp("2023-07-01 01:00:00"), pd.Timestamp("2023-10-01 00:00:00")),
    ("F4 가을", pd.Timestamp("2023-10-01 01:00:00"), pd.Timestamp("2024-01-01 00:00:00")),
]
MIN_TRAIN_ROWS = 1000   # 학습 라벨이 이보다 적으면 그 폴드는 건너뛴다 (그룹 3의 F1)


def fold_masks(feat, group, fold):
    """한 그룹·한 폴드의 (학습, 검증) 마스크. 학습은 검증 시작 이전의 라벨 있는 행 전부(확장 윈도)."""
    _, vs, ve = fold
    has_label = feat[group].notna()
    train = has_label & (feat["forecast_kst_dtm"] < vs)
    val = (feat["forecast_kst_dtm"] >= vs) & (feat["forecast_kst_dtm"] <= ve)
    if int(train.sum()) < MIN_TRAIN_ROWS:
        return None
    return train, val


def scored(feat, group, mask):
    """마스크 중 '채점 대상'(실제 이용률 >= 10%)인 행만 남긴다."""
    return mask & (feat[group] >= CAPACITY_KWH[group] * SCORE_THRESHOLD)


def cv_group(feat, cols, group, params):
    """한 그룹의 폴드 평균 group_score. 학습은 actual 가중·라벨 있는 행 전부(05_tuning의 최적 스위치)."""
    scores = []
    for fold in FOLDS:
        m = fold_masks(feat, group, fold)
        if m is None:
            continue
        tr_m, va_m = m
        dtr = lgb.Dataset(feat.loc[tr_m, cols], label=feat.loc[tr_m, group],
                          weight=feat.loc[tr_m, group].to_numpy())
        sv = scored(feat, group, va_m)
        dva = lgb.Dataset(feat.loc[sv, cols], label=feat.loc[sv, group], reference=dtr)
        booster = lgb.train(params, dtr, MAX_ROUNDS, valid_sets=[dva],
                            callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
        pred = np.clip(booster.predict(feat.loc[va_m, cols], num_iteration=booster.best_iteration),
                       0, CAPACITY_KWH[group])
        scores.append(group_score(feat.loc[va_m, group].to_numpy(), pred, CAPACITY_KWH[group])[0])
    return float(np.mean(scores))


def tune_lgb(feat, cols, tag):
    """그룹마다 (1) tau 격자와 (2) Optuna 20회를 돌려 CV가 높은 쪽을 채택한다."""
    out = {}
    for g in TARGET_COLS:
        # (1) tau 격자
        grid = [(cv_group(feat, cols, g, {**LGB_PARAMS, "alpha": t}), t) for t in TAU_GRID]
        cv_grid, best_tau = max(grid)
        params_grid = {**LGB_PARAMS, "alpha": best_tau}

        # (2) Optuna (탐색 범위는 05_tuning.ipynb §5와 동일)
        def objective(trial):
            p = {"objective": "quantile",
                 "alpha": trial.suggest_float("alpha", 0.45, 0.75),
                 "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.10, log=True),
                 "num_leaves": trial.suggest_int("num_leaves", 15, 127, log=True),
                 "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 120, log=True),
                 "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
                 "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
                 "bagging_freq": 1,
                 "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
                 "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
                 "verbosity": -1, "seed": RNG_SEED, "num_threads": N_THREADS,
                 "deterministic": True, "force_row_wise": True}
            return cv_group(feat, cols, g, p)

        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=RNG_SEED))
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        params_opt = {**study.best_params, "objective": "quantile", "bagging_freq": 1,
                      "verbosity": -1, "seed": RNG_SEED, "num_threads": N_THREADS,
                      "deterministic": True, "force_row_wise": True}

        # (3) CV가 높은 쪽 채택
        if study.best_value > cv_grid:
            chosen, src, cv = params_opt, "optuna", study.best_value
        else:
            chosen, src, cv = params_grid, "tau_grid", cv_grid

        out[g] = {"params": chosen, "source": src, "cv": cv,
                  "cv_grid": cv_grid, "cv_optuna": study.best_value, "tau_grid_best": best_tau}
        print(f"  [{tag}] {g}: tau격자 {cv_grid:.4f}(tau={best_tau}) vs Optuna {study.best_value:.4f} "
              f"-> {src} 채택 (alpha={chosen['alpha']:.3f})")
    return out


def lgb_inner_pred(feat, cols, group, params):
    """내부검증(2023 하반기) 구간 예측. 블렌드 w를 고르는 데 쓴다."""
    m = build_masks(feat, group)
    dtr = lgb.Dataset(feat.loc[m["inner_tr"], cols], label=feat.loc[m["inner_tr"], group],
                      weight=feat.loc[m["inner_tr"], group].to_numpy())
    dva = lgb.Dataset(feat.loc[m["inner_va_scored"], cols],
                      label=feat.loc[m["inner_va_scored"], group], reference=dtr)
    booster = lgb.train(params, dtr, MAX_ROUNDS, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    return np.clip(booster.predict(feat.loc[m["inner_va"], cols],
                                   num_iteration=booster.best_iteration), 0, CAPACITY_KWH[group])


def mlp_inner_pred(feat, cols, group, t_soft):
    """내부검증 구간을 예측하는 MLP를 시드 5개로 학습해 평균 예측과 내부검증 점수를 돌려준다.
    (2단계 전체재학습이 없어 홀드아웃 채점보다 절반 가격이다.)"""
    cap = CAPACITY_KWH[group]
    m = build_masks(feat, group)
    X_itr = feat.loc[m["inner_tr_scored"], cols].to_numpy(np.float32)
    y_itr = (feat.loc[m["inner_tr_scored"], group] / cap).to_numpy(np.float32)
    X_iva_s = feat.loc[m["inner_va_scored"], cols].to_numpy(np.float32)
    y_iva_s = (feat.loc[m["inner_va_scored"], group] / cap).to_numpy(np.float32)
    X_iva = feat.loc[m["inner_va"], cols].to_numpy(np.float32)

    mu, sd = fit_standardizer(X_itr)
    Xi, Xv_s = (X_itr - mu) / sd, standardize(X_iva_s, mu, sd)

    def eval_fn(model):
        with torch.no_grad():
            p = np.clip(model(Xv_s).numpy(), 0, 1)
        return group_score(y_iva_s * cap, p * cap, cap)[0]

    preds = []
    for s in NN_SEEDS:
        model = train_mlp(Xi, y_itr, seed=s, n_epochs=MAX_EPOCHS, t_soft=t_soft, eval_fn=eval_fn)[0]
        preds.append(predict_mlp(model, X_iva, mu, sd, cap))
    pred = np.mean(preds, axis=0)
    act = feat.loc[m["inner_va"], group].to_numpy()
    return pred, group_score(act, pred, cap)[0]


def tune_mlp_and_blend(feat, cols, lgb_tuned, tag):
    """T_soft를 내부검증에서 고르고, 그 T_soft의 예측으로 블렌드 w까지 고른다."""
    # (1) T_soft: 그룹 점수의 평균(=total의 추정)이 가장 높은 값
    tsoft_scores, cache = {}, {}
    for ts in TSOFT_GRID:
        per = {}
        for g in TARGET_COLS:
            cache[(ts, g)] = mlp_inner_pred(feat, cols, g, ts)
            per[g] = cache[(ts, g)][1]
        tsoft_scores[ts] = float(np.mean(list(per.values())))
        print(f"  [{tag}] T_soft={ts:<6} 내부검증 평균 {tsoft_scores[ts]:.4f}  "
              + " ".join(f"g{g[-1]}={per[g]:.4f}" for g in TARGET_COLS))
    best_ts = max(tsoft_scores, key=tsoft_scores.get)

    # (2) 블렌드 w: 이미 만든 예측 재사용 (추가 학습 없음)
    w_best = {}
    for g in TARGET_COLS:
        m = build_masks(feat, g)
        cap = CAPACITY_KWH[g]
        act = feat.loc[m["inner_va"], g].to_numpy()
        p_mlp = cache[(best_ts, g)][0]
        p_lgb = lgb_inner_pred(feat, cols, g, lgb_tuned[g]["params"])
        cand = [(group_score(act, np.clip((1 - w) * p_lgb + w * p_mlp, 0, cap), cap)[0], w) for w in W_GRID]
        s, w = max(cand)
        w_best[g] = float(w)
        print(f"  [{tag}] {g}: 최적 w={w:.2f} (내부검증 {s:.4f})")

    return {"t_soft": best_ts, "w": w_best, "tsoft_scores": tsoft_scores}


print("재튜닝 도구 준비 완료")
print(f"  LightGBM: tau 격자 {len(TAU_GRID)}개 + Optuna {N_TRIALS}회, 폴드 {len(FOLDS)}개")
print(f"  MLP T_soft 후보: {TSOFT_GRID}")
print(f"  블렌드 w 격자: 0.00~1.00, 0.05 간격 ({len(W_GRID)}개)")

재튜닝 도구 준비 완료
  LightGBM: tau 격자 6개 + Optuna 20회, 폴드 4개
  MLP T_soft 후보: [0.003, 0.006, 0.012, 0.025]
  블렌드 w 격자: 0.00~1.00, 0.05 간격 (21개)


**실행 후 확인할 것**

- 함수 정의뿐이라 즉시 끝난다.
- `T_soft` 후보에 현재값 **0.006이 포함**돼 있어야 한다 — 그래야 "재튜닝해도 그대로였다"는 결과가 나올 수 있다.


## 5. v1 기준 재튜닝 — 같은 절차를 v1에도 적용한다

**왜 v1도 다시 튜닝하는가**: 2×2의 C 칸(v1 피처 + v2 튜닝)을 만들려면 두 설정이 **같은 절차**로 나와야 한다.
현재 확정 설정은 과거에 조금씩 다른 방식으로 쌓인 값이라, 그것만으로는 "절차가 같은데 피처만 다른" 비교가 안 된다.

여기서 나온 설정이 확정값(τ 0.70/0.50/0.65, `T_soft` 0.006, w 0.8/0.5/0.9)과 **비슷하게 나오면**
이 튜닝 절차가 과거 결정을 재현한다는 뜻이라 신뢰도가 올라간다.

⏱️ **약 12~15분** (LightGBM CV·Optuna 6분 + MLP T_soft 4후보 8분)


In [6]:
t0 = time.time()
print("=== v1 LightGBM 재튜닝 (2022~2023 CV) ===")
lgb_tuned_v1 = tune_lgb(feat_v1, COLS_V1, "v1")

print("\n=== v1 MLP T_soft · 블렌드 w 재튜닝 (내부검증 2023 하반기) ===")
mlpw_v1 = tune_mlp_and_blend(feat_v1, COLS_V1, lgb_tuned_v1, "v1")

HP_V1_RETUNED = {"lgb": {g: lgb_tuned_v1[g]["params"] for g in TARGET_COLS},
                 "t_soft": mlpw_v1["t_soft"], "w": mlpw_v1["w"]}

print(f"\n--- v1 재튜닝 결과 (소요 {time.time()-t0:.0f}초) ---")
print(f"T_soft: {HP_V1_RETUNED['t_soft']}  (확정값 {T_SOFT})")
for g in TARGET_COLS:
    print(f"  {g}: alpha={HP_V1_RETUNED['lgb'][g]['alpha']:.3f} (확정 {GROUP_TAU[g]}) | "
          f"w={HP_V1_RETUNED['w'][g]:.2f} (확정 {BLEND_W[g]}) | 출처={lgb_tuned_v1[g]['source']}")

=== v1 LightGBM 재튜닝 (2022~2023 CV) ===
  [v1] kpx_group_1: tau격자 0.6030(tau=0.7) vs Optuna 0.6083 -> optuna 채택 (alpha=0.663)
  [v1] kpx_group_2: tau격자 0.6456(tau=0.5) vs Optuna 0.6454 -> tau_grid 채택 (alpha=0.500)
  [v1] kpx_group_3: tau격자 0.5692(tau=0.65) vs Optuna 0.5715 -> optuna 채택 (alpha=0.663)

=== v1 MLP T_soft · 블렌드 w 재튜닝 (내부검증 2023 하반기) ===
  [v1] T_soft=0.003  내부검증 평균 0.6196  g1=0.6255 g2=0.6553 g3=0.5779
  [v1] T_soft=0.006  내부검증 평균 0.6173  g1=0.6238 g2=0.6521 g3=0.5759
  [v1] T_soft=0.012  내부검증 평균 0.6152  g1=0.6217 g2=0.6529 g3=0.5709
  [v1] T_soft=0.025  내부검증 평균 0.6109  g1=0.6171 g2=0.6524 g3=0.5632
  [v1] kpx_group_1: 최적 w=0.65 (내부검증 0.6323)
  [v1] kpx_group_2: 최적 w=0.35 (내부검증 0.6616)
  [v1] kpx_group_3: 최적 w=0.85 (내부검증 0.5795)

--- v1 재튜닝 결과 (소요 727초) ---
T_soft: 0.003  (확정값 0.006)
  kpx_group_1: alpha=0.663 (확정 0.7) | w=0.65 (확정 0.8) | 출처=optuna
  kpx_group_2: alpha=0.500 (확정 0.5) | w=0.35 (확정 0.5) | 출처=tau_grid
  kpx_group_3: alpha=0.663 (확정 0.65) | w=0.85 (확정 0.9) | 출처

**실행 후 확인할 것**

- `alpha`가 확정값(0.70/0.50/0.65)과 **가까운 방향**인지. 크게 다르면 CV와 홀드아웃이 다른 곳을 가리킨다는 신호다.
- `T_soft`가 0.006으로 뽑히면 "현재값이 이미 최적"이라는 독립 확인이 된다.
- `출처`가 `tau_grid`면 05_tuning의 결론(Optuna 코어 재탐색은 단순 τ를 못 이긴다)이 재현된 것이다.


## 6. v2 기준 재튜닝 — 완전히 똑같은 절차를 v2에 적용한다

바뀌는 것은 **입력 피처 목록 하나뿐**이다. 폴드·격자·시도 횟수·시드 전부 §5와 동일하다.

⏱️ **약 12~15분**


In [7]:
t0 = time.time()
print("=== v2 LightGBM 재튜닝 (2022~2023 CV) ===")
lgb_tuned_v2 = tune_lgb(feat_v2, COLS_V2, "v2")

print("\n=== v2 MLP T_soft · 블렌드 w 재튜닝 (내부검증 2023 하반기) ===")
mlpw_v2 = tune_mlp_and_blend(feat_v2, COLS_V2, lgb_tuned_v2, "v2")

HP_V2_RETUNED = {"lgb": {g: lgb_tuned_v2[g]["params"] for g in TARGET_COLS},
                 "t_soft": mlpw_v2["t_soft"], "w": mlpw_v2["w"]}

print(f"\n--- v2 재튜닝 결과 (소요 {time.time()-t0:.0f}초) ---")
print(f"T_soft: {HP_V2_RETUNED['t_soft']}  (v1 재튜닝 {HP_V1_RETUNED['t_soft']})")
for g in TARGET_COLS:
    print(f"  {g}: alpha={HP_V2_RETUNED['lgb'][g]['alpha']:.3f} (v1 재튜닝 {HP_V1_RETUNED['lgb'][g]['alpha']:.3f}) | "
          f"w={HP_V2_RETUNED['w'][g]:.2f} (v1 재튜닝 {HP_V1_RETUNED['w'][g]:.2f}) | 출처={lgb_tuned_v2[g]['source']}")

print(f"\n--- CV 위에서 본 v1 vs v2 (LightGBM 단독, 2022~2023) ---")
for g in TARGET_COLS:
    a, b = lgb_tuned_v1[g]["cv"], lgb_tuned_v2[g]["cv"]
    print(f"  {g}: v1 {a:.4f} -> v2 {b:.4f} ({b-a:+.4f})")
_a = float(np.mean([lgb_tuned_v1[g]["cv"] for g in TARGET_COLS]))
_b = float(np.mean([lgb_tuned_v2[g]["cv"] for g in TARGET_COLS]))
print(f"  평균: v1 {_a:.4f} -> v2 {_b:.4f} ({_b-_a:+.4f})")

=== v2 LightGBM 재튜닝 (2022~2023 CV) ===
  [v2] kpx_group_1: tau격자 0.6018(tau=0.75) vs Optuna 0.6042 -> optuna 채택 (alpha=0.630)
  [v2] kpx_group_2: tau격자 0.6451(tau=0.55) vs Optuna 0.6434 -> tau_grid 채택 (alpha=0.550)
  [v2] kpx_group_3: tau격자 0.5719(tau=0.7) vs Optuna 0.5685 -> tau_grid 채택 (alpha=0.700)

=== v2 MLP T_soft · 블렌드 w 재튜닝 (내부검증 2023 하반기) ===
  [v2] T_soft=0.003  내부검증 평균 0.6201  g1=0.6256 g2=0.6551 g3=0.5796
  [v2] T_soft=0.006  내부검증 평균 0.6188  g1=0.6214 g2=0.6524 g3=0.5826
  [v2] T_soft=0.012  내부검증 평균 0.6126  g1=0.6193 g2=0.6521 g3=0.5665
  [v2] T_soft=0.025  내부검증 평균 0.6123  g1=0.6187 g2=0.6533 g3=0.5650
  [v2] kpx_group_1: 최적 w=0.85 (내부검증 0.6260)
  [v2] kpx_group_2: 최적 w=0.65 (내부검증 0.6593)
  [v2] kpx_group_3: 최적 w=0.95 (내부검증 0.5799)

--- v2 재튜닝 결과 (소요 760초) ---
T_soft: 0.003  (v1 재튜닝 0.003)
  kpx_group_1: alpha=0.630 (v1 재튜닝 0.663) | w=0.85 (v1 재튜닝 0.65) | 출처=optuna
  kpx_group_2: alpha=0.550 (v1 재튜닝 0.500) | w=0.65 (v1 재튜닝 0.35) | 출처=tau_grid
  kpx_group_3: alpha=0.700 (v1 

**실행 후 확인할 것**

- **CV 위의 v1 vs v2 비교**가 여기서 처음 나온다. 이것은 홀드아웃과 **독립된** 증거라 판정에 무게가 있다.
  CV에서도 v2가 지면 홀드아웃 결과와 방향이 일치하는지 §8에서 대조한다.
- `T_soft`나 `w`가 v1과 크게 달라졌다면, 추가 피처가 두 모델의 오차 성격을 바꿨다는 뜻이다 — 해석에 적어 둔다.


## 7. 2×2 채점 — 남은 세 칸을 2024 홀드아웃에서 잰다

A는 §3에서 이미 쟀다. 여기서는 나머지를 채운다.

| 칸 | 피처 | 하이퍼파라미터 | 뜻 |
|---|---|---|---|
| A | v1 | 확정값 | 현 확정 모델 (§3) |
| A′ | v1 | **v1 재튜닝** | 튜닝 절차가 확정값을 재현하는지 확인 |
| B | v2 | 확정값 | 옛 옷을 입은 v2 |
| C | v1 | **v2 재튜닝** | **교란 제거 팔** — 새 설정만의 효과 |
| D | v2 | **v2 재튜닝** | 공정한 v2 |

⏱️ **약 16~20분** (칸 하나당 4~5분)


In [8]:
runs = {}
t0 = time.time()

print("=== 칸 A′: v1 피처 + v1 재튜닝 ===")
runs["A2"] = run_config(feat_v1, COLS_V1, HP_V1_RETUNED, "A2_v1feat_v1retuned")

print("\n=== 칸 B: v2 피처 + v1 확정 설정 ===")
runs["B"] = run_config(feat_v2, COLS_V2, HP_V1, "B_v2feat_v1hp")

print("\n=== 칸 C: v1 피처 + v2 재튜닝 (교란 제거 팔) ===")
runs["C"] = run_config(feat_v1, COLS_V1, HP_V2_RETUNED, "C_v1feat_v2hp")

print("\n=== 칸 D: v2 피처 + v2 재튜닝 ===")
runs["D"] = run_config(feat_v2, COLS_V2, HP_V2_RETUNED, "D_v2feat_v2hp")

runs["A"] = res_A
print(f"\n네 칸 채점 완료 (총 {time.time()-t0:.0f}초)")

=== 칸 A′: v1 피처 + v1 재튜닝 ===
  kpx_group_1: 트리 150그루 | MLP 41에폭 x 시드 5개 | w=0.65
  kpx_group_2: 트리 304그루 | MLP 86에폭 x 시드 5개 | w=0.35
  kpx_group_3: 트리 159그루 | MLP 41에폭 x 시드 5개 | w=0.85

=== 칸 B: v2 피처 + v1 확정 설정 ===
  kpx_group_1: 트리 181그루 | MLP 41에폭 x 시드 5개 | w=0.8
  kpx_group_2: 트리 335그루 | MLP 71에폭 x 시드 5개 | w=0.5
  kpx_group_3: 트리 186그루 | MLP 36에폭 x 시드 5개 | w=0.9

=== 칸 C: v1 피처 + v2 재튜닝 (교란 제거 팔) ===
  kpx_group_1: 트리 315그루 | MLP 41에폭 x 시드 5개 | w=0.85
  kpx_group_2: 트리 477그루 | MLP 86에폭 x 시드 5개 | w=0.65
  kpx_group_3: 트리 206그루 | MLP 41에폭 x 시드 5개 | w=0.95

=== 칸 D: v2 피처 + v2 재튜닝 ===
  kpx_group_1: 트리 147그루 | MLP 51에폭 x 시드 5개 | w=0.85
  kpx_group_2: 트리 536그루 | MLP 86에폭 x 시드 5개 | w=0.65
  kpx_group_3: 트리 184그루 | MLP 41에폭 x 시드 5개 | w=0.95

네 칸 채점 완료 (총 587초)


**실행 후 확인할 것**

- **A′가 A와 비슷한가**(±0.003). 크게 다르면 튜닝 절차가 과거 결정과 어긋난다는 뜻이라, 그 차이를 먼저 해석해야 한다.
- 네 칸 모두 LightGBM·MLP·블렌드가 상식적인 범위(0.60~0.66)인지.


## 8. 효과 분해와 판정 — 이 차이는 신호인가 노이즈인가

숫자가 올랐다고 채택하지 않는다. **이 파이프라인이 가만히 있어도 흔들리는 폭**과 비교해야 한다.

### 노이즈를 어떻게 재는가

MLP는 시드(난수 초기값)마다 다르게 학습된다. **시드 5개 각각으로 블렌드 점수를 따로 내면**
"피처를 하나도 안 바꿔도 생기는 흔들림"의 크기를 알 수 있다.

- `시드별 블렌드 점수의 표준편차` = 시드 하나짜리 모델의 흔들림
- 우리가 쓰는 것은 **시드 5개 평균**이라 흔들림이 √5배 줄어든다 → 그 값을 노이즈 기준으로 삼는다

> **표준편차**: 값들이 평균에서 얼마나 흩어져 있는지. 흩어짐이 클수록 "우연히 그 숫자가 나왔을" 가능성이 크다.

### 판정 규칙 — 축이 두 개다

이 실험은 성격이 다른 질문 두 개에 동시에 답한다. 하나로 뭉뚱그리면 잘못된 행동으로 이어진다.

#### ① 결정 축 — 바꿀 것인가: **D − A′** (그리고 확정 모델 **D − A**)

각자 자기에게 맞춰 풀튜닝한 상태끼리의 맞대결이다. **"v2로 갈아탈 가치가 있나"** 에 답한다.
실제로 채택한다면 v2를 v2용 설정과 함께 쓸 것이므로, 이 비교가 현실의 선택지를 그대로 반영한다.

`D − A`(현 확정 모델과의 비교)도 함께 본다. 최종적으로 넘어야 할 것은 **지금 리더보드에 올라가 있는 모델**이다.

#### ② 기전 축 — 이긴 이유가 무엇인가: **D − C**, **B − A′**

설정을 고정해 놓고 피처만 갈아 끼웠을 때의 효과다. **"피처 21개가 실제로 일을 했나"** 에 답한다.

**왜 이걸 따로 봐야 하는가**: D가 A′를 이겼는데 C(옛 피처 + 새 설정)도 똑같이 이긴다면,
피처는 아무 일도 안 했고 이득은 전부 새 하이퍼파라미터 덕이다.
그럼 올바른 행동은 "v2 채택"이 아니라 **"v1을 유지한 채 설정만 갱신"** 이다 — 피처가 21개 적어 더 단순하고 과적합 위험도 낮다.
이 프로젝트는 실제로 이 상황을 겪었다(헤드라인 +0.0036 중 +0.0024가 설정 덕, 피처 순효과는 +0.0012 노이즈).

#### 두 축을 합친 행동 결정표

| D − A′ (결정) | D − C (기전) | 결론 |
|---|---|---|
| 노이즈 이하 | — | **기각.** 확정 모델 유지 |
| 유의하게 +, 그런데 C − A′ 도 비슷하게 + | 노이즈 이하 | **피처 기각, 설정만 채택 검토** (v1 + v2설정 = C) |
| 유의하게 + | 유의하게 + | **v2 채택 후보.** §9 절제로 기여 묶음을 특정하고 제출로 확인 |

"유의하게"의 기준은 **노이즈의 3배 이상**, 채택까지 가려면 **6배**를 본다.
그리고 크기만이 아니라 **성격**도 본다 — 지금까지 리더보드에서 먹힌 개선은 전부
"손실·목표를 채점 산식에 더 정확히 맞추는" 계열이었고, **피처를 더하는 시도는 7번 연속 실패**했다.
이번 v2는 자유도 추가 계열이라 같은 크기의 신호라도 더 엄격하게 본다.


In [9]:
def blend_from_seed(res, si):
    """시드 하나의 MLP 예측만으로 블렌드를 만들어 total을 낸다."""
    pr = {}
    for g in TARGET_COLS:
        cap = CAPACITY_KWH[g]
        w = res["hp"]["w"][g]
        pr[g] = np.clip((1 - w) * res["pred"]["lgb"][g] + w * res["pred_seeds"][g][si], 0, cap)
    return metric(HOLDOUT_ANS, pd.DataFrame(pr))[0]


LABEL = {"A": "A  v1피처 + 확정설정", "A2": "A' v1피처 + v1재튜닝",
         "B": "B  v2피처 + 확정설정", "C": "C  v1피처 + v2재튜닝",
         "D": "D  v2피처 + v2재튜닝"}

rows, noise = [], {}
for k in ["A", "A2", "B", "C", "D"]:
    r = runs[k]
    seed_tot = np.array([blend_from_seed(r, i) for i in range(len(NN_SEEDS))])
    noise[k] = seed_tot.std(ddof=1) / np.sqrt(len(NN_SEEDS))
    rows.append({"칸": LABEL[k], "피처": r["n_features"],
                 "LGB": round(r["lgb"]["total"], 4), "MLP": round(r["mlp"]["total"], 4),
                 "블렌드": round(r["blend"]["total"], 4),
                 "1-NMAE": round(r["blend"]["one_minus_nmae"], 4),
                 "FICR": round(r["blend"]["ficr"], 4),
                 "시드노이즈": round(noise[k], 5)})
print(pd.DataFrame(rows).to_string(index=False))

tA, tA2, tB = (runs[k]["blend"]["total"] for k in ("A", "A2", "B"))
tC, tD = runs["C"]["blend"]["total"], runs["D"]["blend"]["total"]
TOT = {"A": tA, "A2": tA2, "B": tB, "C": tC, "D": tD}


def cmp(a, b):
    """두 칸의 차이와 '그 비교에 실제로 참여하는 두 칸만의' 표준오차 배수를 돌려준다.

    전역 최대 노이즈를 모든 비교에 쓰면, 그 비교에 끼지도 않은 칸의 흔들림 때문에
    신호가 부당하게 작아 보인다(예: A'−A 비교에 D의 노이즈를 쓰는 것).
    독립인 두 평균의 차이의 표준오차는 sqrt(s_a^2 + s_b^2)이다.
    """
    d = TOT[a] - TOT[b]
    se = float(np.hypot(noise[a], noise[b]))
    return d, se, d / se


print(f"\n{'='*82}")
print("  [결정 축] 각자 풀튜닝한 것끼리의 맞대결 — 바꿀 것인가")
for a, b, note in [("D", "A2", "v2풀튜닝 vs v1풀튜닝"), ("D", "A", "v2풀튜닝 vs 현 확정 모델")]:
    d, se, r = cmp(a, b)
    print(f"    {a:>2} - {b:<2} = {d:+.4f}  (SE {se:.5f}, {r:+5.1f}배)   {note}")
print("  [기전 축] 설정을 고정하고 피처만 교체 — 피처가 일했는가")
for a, b, note in [("D", "C", "v2설정 위에서의 피처 효과"), ("B", "A2", "v1설정 위에서의 피처 효과")]:
    d, se, r = cmp(a, b)
    print(f"    {a:>2} - {b:<2} = {d:+.4f}  (SE {se:.5f}, {r:+5.1f}배)   {note}")
print("  [설정 축] 피처를 고정하고 설정만 교체")
for a, b, note in [("C", "A2", "v1 피처 위에서의 설정 효과"),
                   ("A2", "A", "★ 재튜닝이 현 확정 모델을 넘었는가")]:
    d, se, r = cmp(a, b)
    print(f"    {a:>2} - {b:<2} = {d:+.4f}  (SE {se:.5f}, {r:+5.1f}배)   {note}")
print(f"{'='*82}")

decide, _, r_dec = cmp("D", "A2")      # 결정 축: v2풀튜닝 vs v1풀튜닝
mech, _, r_mech = cmp("D", "C")       # 기전 축: 설정 고정하고 피처만
retune_v1, _, r_ret = cmp("A2", "A")  # 기준선 갱신 축: v1 재튜닝이 확정 모델을 넘었나

# --- 판정 (1): 피처를 채택할 것인가 = 결정 축 ---
if r_dec >= 6 and r_mech >= 6:
    v_feat = "v2 채택 후보 — 결정 축·기전 축 모두 통과."
    a_feat = "§9 절제로 기여 묶음을 특정한 뒤 제출로 확인."
elif r_dec >= 3 and r_mech >= 3:
    v_feat = "보류 — 피처가 일하는 신호는 있으나 약하다."
    a_feat = "§9 묶음별 절제로 어느 묶음이 일하는지 확인한 뒤 결정."
elif r_dec >= 3:
    v_feat = "피처 자체는 불명확 — 결정 축은 이겼는데 기전 축(D−C)이 노이즈다."
    a_feat = "이득의 실체가 설정일 수 있다. 아래 (2)를 함께 볼 것."
else:
    v_feat = "v2 피처 기각 — 각자 풀튜닝해 붙여도 v2가 v1을 유의하게 넘지 못한다."
    a_feat = "v2 피처는 버린다(features_v2_*.parquet 미사용 유지)."

# --- 판정 (2): 재튜닝된 설정이 현 확정 모델을 넘었는가 (피처와 독립된 질문) ---
# ★ 이 검사는 (1)의 결과와 무관하게 항상 한다.
#   피처가 기각돼도 "v1 피처 + 재튜닝 설정"(A')이 확정 모델(A)보다 좋을 수 있고,
#   그렇다면 피처를 안 건드리고 설정만 갱신하는 것이 옳은 행동이다.
if r_ret >= 3:
    v_hp = f"★ 재튜닝 설정 채택 검토 — A'가 확정 모델 A를 {retune_v1:+.4f}({r_ret:+.1f}배)로 넘었다."
    a_hp = ("피처는 그대로 두고 설정만 갱신하는 안(= A' 구성)을 별도 후보로 올린다. "
            "train.ipynb의 GROUP_TAU/BLEND_W/LGB_PARAMS + src/nn.py의 T_SOFT.")
elif r_ret <= -3:
    v_hp = f"⚠ 재튜닝 절차가 확정값보다 나쁘다({retune_v1:+.4f}). 절차 자체를 점검할 것."
    a_hp = "CV·내부검증이 홀드아웃과 어긋나는 원인을 먼저 찾는다. 이 표의 다른 비교도 신뢰도가 떨어진다."
else:
    v_hp = f"재튜닝 설정은 확정값과 동급({retune_v1:+.4f}, {r_ret:+.1f}배) — 확정값이 이미 최적이라는 확인."
    a_hp = "설정 갱신 불필요."

print(f"[피처 판정] {v_feat}")
print(f"     행동  : {a_feat}")
print(f"[설정 판정] {v_hp}")
print(f"     행동  : {a_hp}")

RUN_ABLATION = (r_dec >= 3) and (mech > 0) and (r_mech >= 3)
print(f"\n§9 묶음별 절제 실행 여부: {RUN_ABLATION}")

print("\n--- 그룹별 블렌드 점수 ---")
gr = pd.DataFrame({LABEL[k]: {g: round(runs[k]["per_group"][g], 4) for g in TARGET_COLS}
                   for k in ["A", "A2", "B", "C", "D"]}).T
print(gr.to_string())

              칸  피처    LGB    MLP    블렌드  1-NMAE   FICR   시드노이즈
 A  v1피처 + 확정설정 179 0.6308 0.6435 0.6458  0.8744 0.4172 0.00050
A' v1피처 + v1재튜닝 179 0.6330 0.6455 0.6487  0.8744 0.4231 0.00066
 B  v2피처 + 확정설정 200 0.6280 0.6462 0.6494  0.8762 0.4227 0.00066
C  v1피처 + v2재튜닝 179 0.6314 0.6455 0.6479  0.8749 0.4208 0.00078
D  v2피처 + v2재튜닝 200 0.6352 0.6470 0.6493  0.8761 0.4225 0.00117

  [결정 축] 각자 풀튜닝한 것끼리의 맞대결 — 바꿀 것인가
     D - A2 = +0.0006  (SE 0.00135,  +0.4배)   v2풀튜닝 vs v1풀튜닝
     D - A  = +0.0035  (SE 0.00128,  +2.7배)   v2풀튜닝 vs 현 확정 모델
  [기전 축] 설정을 고정하고 피처만 교체 — 피처가 일했는가
     D - C  = +0.0014  (SE 0.00141,  +1.0배)   v2설정 위에서의 피처 효과
     B - A2 = +0.0007  (SE 0.00093,  +0.8배)   v1설정 위에서의 피처 효과
  [설정 축] 피처를 고정하고 설정만 교체
     C - A2 = -0.0009  (SE 0.00103,  -0.8배)   v1 피처 위에서의 설정 효과
    A2 - A  = +0.0029  (SE 0.00083,  +3.5배)   ★ 재튜닝이 현 확정 모델을 넘었는가
[피처 판정] v2 피처 기각 — 각자 풀튜닝해 붙여도 v2가 v1을 유의하게 넘지 못한다.
     행동  : v2 피처는 버린다(features_v2_*.parquet 미사용 유지).
[설정 판정] ★ 재튜닝 설정 채택 검토 — A'가 확정 모델 A

**실행 후 확인할 것**

- `시드노이즈`가 **0.0005~0.002** 정도면 이 파이프라인의 평소 흔들림과 맞다.
- **CV(§6 마지막 줄)와 홀드아웃(여기)이 같은 방향을 가리키는가.** 둘이 어긋나면 어느 쪽도 믿기 어렵다.
- `RUN_ABLATION`이 `False`면 **§9는 건너뛴다**(20분 절약). 바로 §10 로그 → §11 해석으로 간다.


## 9. 묶음별 절제 — 어느 묶음이 일하는가 (§8에서 신호가 있을 때만)

21개를 한꺼번에 넣으면 "누가 기여했는지" 알 수 없다.
`feature_manifest.json`이 정의해 둔 네 묶음을 **하나씩만 v1에 더해** 각각 재본다.

| 묶음 | 개수 | 가설 |
|---|---:|---|
| `icing_persistence` | 6 | 얼음은 착빙 조건이 끝난 뒤에도 날개에 남아 출력을 깎는다 |
| `cloud_fog_icing` | 6 | 착빙은 구름·안개 속에서 일어난다 |
| `radiation_stability` | 6 | 야간 복사냉각 → 안정 경계층 → 강한 시어 |
| `veer` | 3 | 높이에 따른 풍향 변화가 날개 전체의 받음각을 어긋나게 한다 |

> **절제 실험(ablation)**: 한 부품씩만 넣거나 빼서 그 부품의 기여를 따로 재는 것.
> 요리에서 재료 하나씩 빼 보며 맛의 원인을 찾는 것과 같다.

**설정은 v2 재튜닝(HP_V2_RETUNED)으로 고정**한다. 묶음마다 또 재튜닝하면 비용이 4배가 되고,
비교 기준도 C 칸(v1 + 같은 설정)이라 이미 교란이 제거돼 있다.

⏱️ **묶음당 4~5분, 네 묶음이면 16~20분.** `RUN_ABLATION`이 `False`면 이 셀은 아무것도 하지 않는다.


In [10]:
if not RUN_ABLATION:
    print("§8 판정이 기각이므로 절제 실험을 건너뛴다 (RUN_ABLATION=False).")
    ablation = {}
else:
    ablation = {}
    for blk, feats in V2_BLOCKS.items():
        cols = COLS_V1 + feats
        print(f"\n--- {blk} (+{len(feats)}개) ---")
        ablation[blk] = run_config(feat_v2, cols, HP_V2_RETUNED, f"v1+{blk}", verbose=False)
        d = ablation[blk]["blend"]["total"] - runs["C"]["blend"]["total"]
        print(f"  블렌드 {ablation[blk]['blend']['total']:.4f} ({d:+.4f} vs C)")

    base_c = runs["C"]["blend"]["total"]
    print(f"\n{'묶음':<24}{'피처':>6}{'블렌드':>10}{'vs C':>10}")
    print(f"{'C (v1, 같은 설정)':<24}{len(COLS_V1):>6}{base_c:>10.4f}{0.0:>10.4f}")
    for blk in V2_BLOCKS:
        r = ablation[blk]
        print(f"{blk:<24}{r['n_features']:>6}{r['blend']['total']:>10.4f}{r['blend']['total'] - base_c:>+10.4f}")
    print(f"{'D (v2 전부)':<24}{runs['D']['n_features']:>6}{runs['D']['blend']['total']:>10.4f}"
          f"{runs['D']['blend']['total'] - base_c:>+10.4f}")

§8 판정이 기각이므로 절제 실험을 건너뛴다 (RUN_ABLATION=False).


**실행 후 확인할 것**

- 묶음별 이득의 **합**이 D의 이득과 비슷한가? 크게 다르면 묶음끼리 상호작용이 있다는 뜻이다.
- 한 묶음만 크게 기여한다면, **그 묶음만 켜는 것**이 21개 전부 켜는 것보다 나을 수 있다(자유도가 적으니).


## 10. 실험 로그 기록

`experiments/log.csv`에 이어 붙인다. `public_score`는 실제 제출 후 손으로 채운다.


In [11]:
def git_state():
    """읽기 전용 git 명령만 쓴다 (CLAUDE.md 4번 — 커밋은 민석님이 직접 실행)."""
    try:
        h = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
        d = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
        return f"{h}({'dirty' if d else 'clean'})"
    except Exception:
        return "unknown"


def metric_by_group(answer_df, pred_df):
    """metric()과 같은 산식이되, 3개 그룹을 평균내기 전 단계의 그룹별 nmae/ficr을 돌려준다."""
    out = {}
    for g in TARGET_COLS:
        a = answer_df[g].to_numpy(dtype=float)
        f = pred_df[g].to_numpy(dtype=float)
        cap = CAPACITY_KWH[g]
        v = a >= cap * SCORE_THRESHOLD
        a, f = a[v], f[v]
        er = np.abs(f - a) / cap
        price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
        out[g] = {"nmae": float(np.mean(er)), "ficr": float(np.sum(a * price) / np.sum(a * 4.0))}
    return out


LOG_PATH = EXP_DIR / "log.csv"
log = pd.read_csv(LOG_PATH, encoding="utf-8-sig")
next_id = max(int(s[3:]) for s in log["exp_id"] if str(s).startswith("exp")) + 1

to_log = [runs[k] for k in ["A2", "B", "C", "D"]] + [ablation[b] for b in V2_BLOCKS if b in ablation]
new_rows = []
for res in to_log:
    bg = metric_by_group(HOLDOUT_ANS, pd.DataFrame(res["pred"]["blend"]))
    hp = res["hp"]
    # f-string 안에 같은 따옴표를 겹쳐 쓰지 않도록 미리 문자열로 만들어 둔다
    alpha_str = "/".join("{:.2f}".format(hp["lgb"][g]["alpha"]) for g in TARGET_COLS)
    w_str = "/".join("{:.2f}".format(hp["w"][g]) for g in TARGET_COLS)
    row = {"exp_id": f"exp{next_id:03d}", "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
           "git_hash": git_state(),
           "model": "blend(lightgbm_quantile + mlp_metric_loss)",
           "features": res["name"],
           "total_score": round(res["blend"]["total"], 6),
           "one_minus_nmae": round(res["blend"]["one_minus_nmae"], 6),
           "ficr": round(res["blend"]["ficr"], 6),
           "val_period": "2024-01-01~2024-12-31",
           "fit_seconds": round(res["elapsed"], 1),
           "public_score": "",
           "note": (f"06_feature_ablation 2x2 | 피처 {res['n_features']}개 | "
                    f"T_soft={hp['t_soft']} | alpha={alpha_str} | w={w_str} | "
                    f"LGB단독 {res['lgb']['total']:.4f} MLP단독 {res['mlp']['total']:.4f}")}
    for g in TARGET_COLS:
        row[f"nmae_g{g[-1]}"] = round(bg[g]["nmae"], 6)
        row[f"ficr_g{g[-1]}"] = round(bg[g]["ficr"], 6)
    new_rows.append(row)
    next_id += 1

new_df = pd.DataFrame(new_rows)
unknown = [c for c in new_df.columns if c not in log.columns]
assert not unknown, f"log.csv에 없는 컬럼을 만들려 합니다: {unknown}"

log_out = pd.concat([log, new_df], ignore_index=True)[log.columns]
log_out.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
print(f"{len(new_rows)}개 행 추가 -> {LOG_PATH}")
print(new_df[["exp_id", "features", "total_score", "one_minus_nmae", "ficr"]].to_string(index=False))

4개 행 추가 -> d:\공모전\wind_forecast\experiments\log.csv
exp_id            features  total_score  one_minus_nmae     ficr
exp029 A2_v1feat_v1retuned     0.648721        0.874377 0.423065
exp030       B_v2feat_v1hp     0.649422        0.876165 0.422679
exp031       C_v1feat_v2hp     0.647862        0.874884 0.420840
exp032       D_v2feat_v2hp     0.649306        0.876118 0.422495


**실행 후 확인할 것**

- `exp018`부터 번호가 이어져야 한다(기존 마지막이 exp017).
- 칸 A는 기록하지 않는다 — 이미 exp017로 등록된 현 확정 모델이라 중복이다.
- **이 셀은 여러 번 돌리면 행이 중복으로 쌓인다.** 한 번만 실행한다.


## 11. 종합 해석

**두 줄 요약: v2 피처 21개는 기각. 그런데 피처를 하나도 안 바꾸고 재튜닝만 한 A′가 현 확정 모델을 +0.0029(3.5배)로 넘었다.**

### 11-1. 측정 도구는 믿을 수 있다

칸 A의 LightGBM이 `0.6308`, 블렌드가 `0.6458`로 **`reports/train.md` 기록값과 소수점 네 자리까지 정확히 일치**했다(표의 "기록값 대비" 열이 전부 ±0.0000). MLP까지 포함해 재현된 것이라, 아래 비교들은 같은 저울에서 잰 값이다.

### 11-2. 결과표

| 칸 | 피처 | 설정 | 블렌드 | 1−NMAE | FICR | 시드노이즈 |
|---|---:|---|---:|---:|---:|---:|
| A | 179 | 확정값 | 0.6458 | 0.8744 | 0.4172 | 0.00050 |
| **A′** | 179 | **v1 재튜닝** | **0.6487** | 0.8744 | **0.4231** | 0.00066 |
| B | 200 | 확정값 | 0.6494 | 0.8762 | 0.4227 | 0.00066 |
| C | 179 | v2 재튜닝 | 0.6479 | 0.8749 | 0.4208 | 0.00078 |
| D | 200 | v2 재튜닝 | 0.6493 | 0.8761 | 0.4225 | 0.00117 |

| 비교 | 값 | SE | 배수 | 뜻 |
|---|---:|---:|---:|---|
| D − A′ | +0.0006 | 0.00134 | **+0.4** | 결정 축 — 풀튜닝끼리 붙으면 v2가 v1을 **못 이긴다** |
| D − C | +0.0014 | 0.00141 | +1.0 | 기전 축 — 피처 순효과도 노이즈 |
| B − A′ | +0.0007 | 0.00093 | +0.7 | 다른 설정 위에서도 마찬가지 |
| C − A′ | −0.0008 | 0.00102 | −0.8 | v2용 설정을 v1에 씌우면 오히려 손해(당연) |
| **A′ − A** | **+0.0029** | 0.00083 | **+3.5** | **재튜닝만으로 확정 모델을 넘었다** |

### 11-3. v2 피처 — 기각

`veer`가 현 179피처에 0개이고 복사·안정도 지표도 사실상 없어서 "빈칸을 메운다"는 기대가 있었지만, **각자에게 맞춰 풀튜닝한 뒤 붙이면 차이가 노이즈의 0.4배**다. 기전 축(D−C, B−A′)도 1배 안쪽이라 방향이 일관된다. 피처 실험 8번째 실패로 기록한다.

한 가지는 짚어 둔다 — v2가 **1−NMAE는 실제로 올렸다**(0.8744 → 0.8761). 다만 FICR이 그만큼 내려(0.4231 → 0.4225) 총점에서 상쇄됐다. 추가 피처가 예측 정확도 자체엔 조금 기여했으나, 이 대회 산식에서 점수를 만드는 건 **6%/8% 밴드 안에 들어가느냐**이고 거기엔 도움이 안 됐다는 뜻이다.

### 11-4. 진짜 발견 — 하이퍼파라미터가 아직 남아 있었다

A′는 v1 피처 그대로에 설정만 바꾼 것이다.

| | 확정값(A) | 재튜닝(A′) |
|---|---|---|
| `T_soft` | 0.006 | **0.003** |
| g1 τ / w | 0.70 / 0.80 | 0.663 / **0.65** |
| g2 τ / w | 0.50 / 0.50 | 0.50 / **0.35** |
| g3 τ / w | 0.65 / 0.90 | 0.663 / **0.85** |

**세 가지가 이 신호를 믿을 만하게 만든다.**

1. **1−NMAE는 0.8744로 완전히 그대로인데 FICR만 +0.0059 올랐다.** `T_soft`를 0.006 → 0.003으로 낮춘 것은 계단 함수를 더 날카롭게 근사하는 것이고, 그 직접적 타깃이 FICR이다. **원인과 결과가 정확히 맞는다.** 이 프로젝트에서 리더보드로 검증된 개선(τ → FICR 손실 → 그룹별 손실)이 전부 이 "산식을 더 정확히 겨냥" 계열이었다.
2. **A′의 설정은 2024를 한 번도 보지 않고 골랐다**(LightGBM은 2022~2023 4폴드 CV, `T_soft`·w는 2023 하반기 내부검증). 반면 **확정값의 w는 과거에 이 홀드아웃 위에서 고른 값**이라 A 쪽이 유리한 조건이었다. 그런데도 졌다.
3. 그룹별로 보면 **g1이 0.6512 → 0.6600(+0.0088)** 으로 대부분을 만들었다(g2 −0.0026, g3 +0.0025). g1은 확정값에서 τ=0.70·w=0.80으로 가장 세게 밀어 올리던 그룹인데, 그 강도를 낮춘 것이 맞았다.

### 11-5. `T_soft`가 격자 경계에서 멈췄다 — 아직 더 있을 수 있다

후보가 `[0.003, 0.006, 0.012, 0.025]`였는데 **v1·v2 둘 다 최솟값 0.003을 골랐다.** 봉우리가 격자 안에 갇히지 않았다는 뜻이라, **더 작은 값이 더 좋을 가능성이 열려 있다.** 구v5 때도 `T_soft` 격자를 넓혀서 봉우리를 찾은 전례가 있다(그때는 위쪽으로 확장해 0.035에서 정점).

다음 실행 후보: `TSOFT_GRID = [0.0005, 0.001, 0.002, 0.003, 0.004]`로 아래를 열어 다시 훑는다. MLP 내부검증만 돌리면 되므로 **약 15~20분**이면 된다.

### 11-6. 이번에 고친 판정 로직 버그 (정직한 기록)

처음 실행 때 §8이 **"기각 — 확정 모델 유지. train/inference/src 무변경"** 을 출력했는데, **이건 틀린 판정이었다.** 판정 분기가 `D − A′`(결정 축)를 먼저 보고 3배 미만이면 그 자리에서 끝나도록 짜여 있어서, **`A′ > A`인지를 아예 검사하지 않았다.** "피처 기각 / 설정 채택" 갈래를 결정 축 통과 이후에만 도달하게 배치한 설계 실수다.

고친 내용 두 가지:
- **판정을 (1) 피처 / (2) 설정 두 개로 완전히 분리**했다. (2)는 (1)의 결과와 무관하게 **항상** 검사한다. 피처가 기각돼도 기준선 자체가 갱신될 수 있기 때문이다.
- **노이즈를 전역 최댓값에서 쌍별 표준오차로 바꿨다.** 전에는 모든 비교에 가장 큰 흔들림(D칸 0.00117)을 썼는데, `A′ − A` 비교에는 D가 끼지도 않는다. 독립인 두 평균 차이의 표준오차 `sqrt(s_a² + s_b²)`를 쓰도록 고쳤다. 이 하나로 `A′ − A`가 2.5배 → **3.5배**가 됐다(문턱 3배를 넘김).

*(수정된 셀을 다시 실행하면 위 판정이 출력으로 반영된다. 학습은 다시 하지 않으므로 커널이 살아 있다면 §8 셀만 재실행하면 된다.)*

### 11-7. 다음 행동

| 순 | 할 일 | 비용 | 이유 |
|---|---|---|---|
| 1 | **`T_soft` 격자 하방 확장** | 15~20분 | 경계에서 멈췄다(11-5). A′를 더 밀어올릴 여지 |
| 2 | **A′(+확장 결과) 구성을 제출** | 재학습+제출 | 홀드아웃 +0.0029/3.5배, 성격도 검증된 계열. **리더보드로 확인할 가치가 충분하다** |
| 3 | 그다음 구조 레버(g3 통합 학습)로 | ~40분 | HANDOFF 권장 순서 참고 |

**v2 피처는 여기서 종료한다.** `features_v2_*.parquet`는 그대로 두되(재현성), `train.ipynb`/`inference.ipynb`는 v1을 계속 쓴다.


## 12. `T_soft` 격자 하방 확장 — §11-5가 남긴 숙제

여기부터는 위 2×2 실험이 **끝나면서 스스로 지목한 후속 실험**이다. 결론(§11)은 그대로 두고, 그 결론이 "다음에 이걸 해봐야 한다"고 적어 둔 것을 이어서 한다.

### 12-0. 무엇을 / 왜 / 어떻게

**무엇을**: MLP 손실의 `T_soft` 후보를 `[0.003, 0.006, 0.012, 0.025]`에서 **`[0.0005, 0.001, 0.002, 0.003, 0.004]`로 아래쪽을 열어** 다시 훑는다.

**왜**: §5·§6에서 v1도 v2도 **격자의 최솟값 0.003을 골랐다.** 이건 "0.003이 최적"이라는 뜻이 아니라 **"우리가 0.003 아래를 안 봤다"** 는 뜻이다.

> **격자 경계에서 멈춘다는 것**: 산 정상을 찾는데 발판 네 개(0.003·0.006·0.012·0.025)만 놓고 재 봤더니 **맨 끝 발판이 제일 높았다**는 상황이다. 그 너머에 더 높은 곳이 있는지, 아니면 바로 낭떠러지인지는 발판을 더 놓아 보기 전엔 모른다. 이 프로젝트는 구v5 때 같은 상황에서 **위쪽으로** 격자를 넓혀 봉우리를 찾은 전례가 있다(그때 정점은 0.035였다).

그리고 §11-4에서 봤듯 A′의 이득은 **1−NMAE는 그대로인데 FICR만 +0.0059** 오른 형태였다. `T_soft`를 낮춘 것의 직접 타깃이 FICR이므로, **이 레버가 이번 개선의 주범일 가능성이 크다.** 그렇다면 더 밀어 볼 값이 있다.

### 12-1. 그런데 무한정 낮추면 안 되는 이유 (기전)

`T_soft`를 낮추면 두 가지가 **동시에** 일어난다. 하나는 좋고 하나는 나쁘다.

`p_soft(e) = 3·σ((0.08 − e)/T) + σ((0.06 − e)/T)` 에서,

| | 좋은 쪽 | 나쁜 쪽 |
|---|---|---|
| T가 작아지면 | 진짜 계단(4/3/0)에 **더 가까워진다** → 손실이 채점 산식과 더 일치 | 기울기가 **밴드 경계 근처에서만** 살아 있고 나머지는 0에 가까워진다 |

두 번째를 조금 더 풀면 이렇다. 시그모이드의 미분은 경계에서 멀어지면 **지수적으로** 죽는다. 폭이 대략 `±T`인 좁은 창 안에 있는 샘플만 "밴드로 당겨라"는 신호를 받고, 나머지 샘플에게 손실은 사실상 그냥 L1이 된다. `T=0.0005`면 그 창은 이용률 ±0.0005 — group_1 기준 **±10.8 kWh**(설비 21,600 kWh의 0.05%)다. 그 안에 들어오는 학습 샘플이 몇 개나 되겠는가.

게다가 창 안에서는 기울기 크기가 `1/T`로 커진다. `GRAD_CLIP=1.0`(기울기 노름 상한)에 계속 걸리면 **학습이 몇 개 샘플에 끌려다니게** 된다.

**그래서 아래 어딘가에 반드시 봉우리가 있다.** 0.003이 이미 그 봉우리일 수도 있고, 더 아래일 수도 있다. 이 절은 그걸 확인한다.

> **이해 체크**: "T를 낮추면 손실이 산식에 더 가까워지지만, 기울기를 받는 샘플 수가 줄어 학습이 불안정해진다" — 이 한 문장이 되면 OK.

### 12-2. 어떻게 재는가 — §4의 도구를 그대로 쓴다

**2024는 여전히 보지 않는다.** `T_soft` 선택은 §4-2와 똑같이 **내부검증(2023 하반기)** 에서만 한다. 고른 뒤에 딱 한 번 2024 홀드아웃으로 채점한다(그 결과가 아래 **칸 A″**).

| 단계 | 하는 일 | 데이터 | 비용 |
|---|---|---|---|
| 12-3 | 준비 — §5의 v1 재튜닝 LightGBM 설정 확보 | — | 0분(또는 6분) |
| 12-4 | 확장 격자 5개를 내부검증에서 스윕 | 2022~2023 | 약 8~12분 |
| 12-5 | 최적 `T_soft`에서 w 재선택 → 2024 홀드아웃 채점 | 2024는 채점만 | 약 4분 |

**공짜로 얻는 검증 하나**: 확장 격자에 **0.003을 일부러 포함**했다. §5에서 이미 잰 값(내부검증 평균 0.6196)이 그대로 다시 나와야 한다. 안 나오면 이 절의 다른 숫자도 믿을 수 없다는 뜻이다.


In [12]:
# ---------- 12-3. 준비: §5의 v1 재튜닝 LightGBM 설정을 확보한다 ----------
# 이 절은 §5(12분)를 다시 돌리지 않아도 되게 만든다. 세 갈래로 알아서 처리한다.
#   (1) 커널이 살아 있으면  -> 메모리의 HP_V1_RETUNED를 쓰고, 다음을 위해 파일로도 남긴다
#   (2) 커널이 죽었지만 파일이 있으면 -> 읽어서 쓴다 (재계산 없음)
#   (3) 둘 다 없으면        -> LightGBM 재튜닝만 다시 돌린다 (약 6분)
#                              시드가 고정돼 있어 §5와 같은 값이 나온다.
HP_PATH = EXP_DIR / "hp_v1_retuned.json"

if "HP_V1_RETUNED" in globals():
    HP_PATH.write_text(json.dumps(HP_V1_RETUNED, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"(1) 메모리의 §5 결과 사용 -> {HP_PATH.name}에 저장했습니다")
elif HP_PATH.exists():
    HP_V1_RETUNED = json.loads(HP_PATH.read_text(encoding="utf-8"))
    print(f"(2) {HP_PATH.name}에서 §5 결과를 읽었습니다 (재계산 없음)")
else:
    print("(3) §5 결과가 없습니다. LightGBM 재튜닝만 다시 돌립니다 (약 6분)")
    lgb_tuned_v1 = tune_lgb(feat_v1, COLS_V1, "v1")
    # t_soft·w는 어차피 12-4/12-5에서 다시 고르므로 여기서는 §5 출력에 적힌 값을 자리채움으로 둔다.
    HP_V1_RETUNED = {"lgb": {g: lgb_tuned_v1[g]["params"] for g in TARGET_COLS},
                     "t_soft": 0.003,
                     "w": {"kpx_group_1": 0.65, "kpx_group_2": 0.35, "kpx_group_3": 0.85}}
    HP_PATH.write_text(json.dumps(HP_V1_RETUNED, ensure_ascii=False, indent=2), encoding="utf-8")

# 확보한 설정이 §5 기록(alpha 0.663/0.500/0.663)과 맞는지 눈으로 확인한다
print("\n--- 확보된 v1 재튜닝 LightGBM 설정 ---")
for g in TARGET_COLS:
    p = HP_V1_RETUNED["lgb"][g]
    print(f"  {g}: alpha={p['alpha']:.3f}  lr={p['learning_rate']:.4f}  leaves={p['num_leaves']}  "
          f"min_leaf={p['min_data_in_leaf']}")
print(f"  (§5 당시 선택: T_soft={HP_V1_RETUNED['t_soft']}, w={HP_V1_RETUNED['w']})")

(1) 메모리의 §5 결과 사용 -> hp_v1_retuned.json에 저장했습니다

--- 확보된 v1 재튜닝 LightGBM 설정 ---
  kpx_group_1: alpha=0.663  lr=0.0607  leaves=30  min_leaf=107
  kpx_group_2: alpha=0.500  lr=0.0500  leaves=63  min_leaf=40
  kpx_group_3: alpha=0.663  lr=0.0688  leaves=35  min_leaf=107
  (§5 당시 선택: T_soft=0.003, w={'kpx_group_1': 0.65, 'kpx_group_2': 0.35, 'kpx_group_3': 0.85})


**실행 후 확인할 것**

- 세 갈래 중 어느 쪽으로 갔는지 `(1)/(2)/(3)` 표시로 확인한다.
- `alpha`가 **0.663 / 0.500 / 0.663** 이어야 한다(§5 출력과 동일). 다르면 §5를 다시 돌려야 한다.
- `experiments/hp_v1_retuned.json`이 생겼는지 확인한다 — 앞으로의 실험(g3 통합 학습 등)이 이 파일을 재사용한다.


### 12-4. 확장 격자 스윕

`[0.0005, 0.001, 0.002, 0.003, 0.004]` 다섯 값 각각에 대해 **MLP를 시드 5개씩 새로 학습**해 내부검증(2023 하반기) 점수를 낸다. 그룹 3개 × 5시드 × 5값 = MLP 75번 학습이라 이 절에서 가장 비싼 부분이다.

⏱️ **약 8~12분**

§5에서 이미 잰 위쪽 값들(0.006 / 0.012 / 0.025)을 붙여 **곡선 전체 모양**을 함께 본다. 봉우리가 안쪽에 갇혔는지 한눈에 보려는 것이다.


In [13]:
# ---------- 12-4. T_soft 확장 격자 스윕 (내부검증에서만) ----------
TSOFT_GRID_EXT = [0.0005, 0.001, 0.002, 0.003, 0.004]

# §5의 v1 실행에서 이미 얻은 값 (위쪽 격자). 곡선 모양을 보려고 붙인다.
TSOFT_PREV_V1 = {0.003: 0.6196, 0.006: 0.6173, 0.012: 0.6152, 0.025: 0.6109}

t0 = time.time()
ext_cache, ext_scores, ext_per = {}, {}, {}
for ts in TSOFT_GRID_EXT:
    per = {}
    for g in TARGET_COLS:
        ext_cache[(ts, g)] = mlp_inner_pred(feat_v1, COLS_V1, g, ts)   # (예측, 그룹점수)
        per[g] = ext_cache[(ts, g)][1]
    ext_per[ts] = per
    ext_scores[ts] = float(np.mean(list(per.values())))
    print(f"  T_soft={ts:<7} 내부검증 평균 {ext_scores[ts]:.4f}  "
          + " ".join(f"g{g[-1]}={per[g]:.4f}" for g in TARGET_COLS))
print(f"\n스윕 소요 {time.time() - t0:.0f}초")

# --- (a) 재현 확인: 0.003은 §5에서 이미 쟀다. 같은 값이 나와야 한다 ---
d_repro = ext_scores[0.003] - TSOFT_PREV_V1[0.003]
print(f"\n[재현 확인] T_soft=0.003 : 이번 {ext_scores[0.003]:.4f} vs §5 {TSOFT_PREV_V1[0.003]:.4f} "
      f"({d_repro:+.4f}) -> {'일치' if abs(d_repro) < 5e-5 else '★불일치 — 아래 숫자를 믿지 말 것'}")

# --- (b) 곡선 전체 (확장 + §5의 위쪽 값) ---
curve = {**TSOFT_PREV_V1, **ext_scores}          # 0.003은 이번 값으로 덮어쓴다
print("\n--- T_soft 곡선 (내부검증 평균, 낮은 값 -> 높은 값 순) ---")
BEST_TS = max(ext_scores, key=ext_scores.get)
for ts in sorted(curve):
    bar = "#" * int(round((curve[ts] - 0.605) * 2000))
    mark = "  <- 확장 격자 최적" if ts == BEST_TS else ("  (§5)" if ts not in ext_scores else "")
    print(f"  {ts:<7} {curve[ts]:.4f} {bar}{mark}")

# --- (c) 봉우리가 격자 안에 갇혔는가 ---
lo, hi = min(TSOFT_GRID_EXT), max(TSOFT_GRID_EXT)
if BEST_TS == lo:
    verdict = ("★ 또 경계(최솟값)에서 멈췄다. 12-1의 기전상 더 낮추면 언젠가 무너져야 하는데 "
               "아직 안 무너졌다는 뜻이다. 아래를 한 번 더 열어 볼 가치가 있다.")
elif BEST_TS == hi:
    verdict = "★ 위쪽 경계다. 0.004~0.006 사이를 더 촘촘히 볼 것."
else:
    verdict = "✔ 봉우리가 격자 안에 갇혔다. 양옆이 모두 더 낮으므로 이 값을 최적으로 본다."
print(f"\n[경계 판정] 최적 T_soft = {BEST_TS}\n  {verdict}")

# --- (d) 그룹별로 봉우리 위치가 다른가 (= '그룹별 T_soft' 후보의 사전 신호) ---
print("\n--- 그룹별 최적 T_soft (공통값 하나로 묶는 게 손해인지 미리 본다) ---")
for g in TARGET_COLS:
    best_g = max(TSOFT_GRID_EXT, key=lambda t: ext_per[t][g])
    gain = ext_per[best_g][g] - ext_per[BEST_TS][g]
    print(f"  {g}: 최적 {best_g:<7} (공통 {BEST_TS} 대비 {gain:+.4f})")

IMPROVED = ext_scores[BEST_TS] > ext_scores[0.003] + 1e-9
print(f"\n0.003보다 나은 값을 찾았는가: {IMPROVED} "
      f"({ext_scores[BEST_TS] - ext_scores[0.003]:+.4f})")

  T_soft=0.0005  내부검증 평균 0.6256  g1=0.6332 g2=0.6540 g3=0.5895
  T_soft=0.001   내부검증 평균 0.6265  g1=0.6332 g2=0.6556 g3=0.5907
  T_soft=0.002   내부검증 평균 0.6203  g1=0.6250 g2=0.6573 g3=0.5785
  T_soft=0.003   내부검증 평균 0.6196  g1=0.6255 g2=0.6553 g3=0.5779
  T_soft=0.004   내부검증 평균 0.6173  g1=0.6213 g2=0.6533 g3=0.5773

스윕 소요 466초

[재현 확인] T_soft=0.003 : 이번 0.6196 vs §5 0.6196 (-0.0000) -> 일치

--- T_soft 곡선 (내부검증 평균, 낮은 값 -> 높은 값 순) ---
  0.0005  0.6256 #########################################
  0.001   0.6265 ###########################################  <- 확장 격자 최적
  0.002   0.6203 ###############################
  0.003   0.6196 #############################
  0.004   0.6173 #########################
  0.006   0.6173 #########################  (§5)
  0.012   0.6152 ####################  (§5)
  0.025   0.6109 ############  (§5)

[경계 판정] 최적 T_soft = 0.001
  ✔ 봉우리가 격자 안에 갇혔다. 양옆이 모두 더 낮으므로 이 값을 최적으로 본다.

--- 그룹별 최적 T_soft (공통값 하나로 묶는 게 손해인지 미리 본다) ---
  kpx_group_1: 최적 0.0005  (공통 0.001 대비 +

**실행 후 확인할 것**

- **[재현 확인]이 "일치"** 여야 한다. 여기서 어긋나면 아래를 볼 필요가 없다.
- 곡선이 **아래로 내려가면서 꺾이는지**(봉우리가 갇힘) 아니면 계속 오르는지(또 경계).
- 그룹별 최적이 서로 많이 다르면 → HANDOFF 5-(b) "그룹별 `T_soft`"의 사전 신호다. 여기서 결정하진 않고 관찰만 적어 둔다.
- `IMPROVED`가 `False`면 12-5는 **A′와 똑같은 구성을 다시 재는 것**이라 건너뛴다(셀이 알아서 판단한다).


### 12-5. 칸 A″ — 최적 `T_soft`로 2024 홀드아웃 채점

12-4가 고른 `T_soft`에서 **블렌드 가중치 w를 다시 고른다**(§4-3과 같은 방식, 이미 만든 내부검증 예측을 재사용하므로 추가 학습 거의 없음). 그다음 그 구성으로 2022~2023을 학습해 **2024를 딱 한 번 채점**한다.

이렇게 나온 칸을 **A″**라 부르고 두 가지와 비교한다.

| 비교 | 묻는 것 |
|---|---|
| **A″ − A′** | 격자를 넓힌 게 실제로 값을 했는가 |
| **A″ − A** | 현 확정 모델(exp017, Public 0.63886) 대비 총 이득 — **제출 판단의 근거** |

노이즈 기준은 §8과 같다. 시드 5개 각각으로 블렌드를 만들어 표준오차를 재고, **쌍별로** `sqrt(s_a² + s_b²)`를 쓴다.

⏱️ **약 4~5분**


In [14]:
# ---------- 12-5. 최적 T_soft에서 w 재선택 -> 2024 홀드아웃 채점 (칸 A'') ----------
if not IMPROVED:
    print("12-4에서 0.003을 넘는 값이 없었다 -> A''는 A'와 같은 구성이므로 채점을 건너뛴다.")
    print("결론: T_soft 확장은 개선 없음. A' 구성(T_soft=0.003)을 그대로 제출 후보로 유지한다.")
else:
    # --- (1) 블렌드 w 재선택 (§4-3과 동일 절차, 내부검증에서만) ---
    print(f"=== w 재선택 (T_soft={BEST_TS}, 내부검증 2023 하반기) ===")
    w_ext = {}
    for g in TARGET_COLS:
        m = build_masks(feat_v1, g)
        cap = CAPACITY_KWH[g]
        act = feat_v1.loc[m["inner_va"], g].to_numpy()
        p_mlp = ext_cache[(BEST_TS, g)][0]
        p_lgb = lgb_inner_pred(feat_v1, COLS_V1, g, HP_V1_RETUNED["lgb"][g])
        cand = [(group_score(act, np.clip((1 - w) * p_lgb + w * p_mlp, 0, cap), cap)[0], w)
                for w in W_GRID]
        s, w = max(cand)
        w_ext[g] = float(w)
        print(f"  {g}: 최적 w={w:.2f} (내부검증 {s:.4f})  [A' 때는 {HP_V1_RETUNED['w'][g]:.2f}]")

    HP_A3 = {"lgb": HP_V1_RETUNED["lgb"], "t_soft": BEST_TS, "w": w_ext}

    # --- (2) 2024 홀드아웃 채점 ---
    print(f"\n=== 칸 A'': v1 피처 + v1 재튜닝 + T_soft {BEST_TS} ===")
    if "runs" not in globals():
        runs = {}
    runs["A3"] = run_config(feat_v1, COLS_V1, HP_A3, f"A3_v1feat_tsoft{BEST_TS}")
    rA3 = runs["A3"]
    print(f"  블렌드 {rA3['blend']['total']:.4f} "
          f"(1-NMAE {rA3['blend']['one_minus_nmae']:.4f}, FICR {rA3['blend']['ficr']:.4f}) "
          f"| 소요 {rA3['elapsed']:.0f}초")

    # --- (3) A'/A와 비교 ---
    def _blend_from_seed(res, si):
        """시드 하나의 MLP 예측만으로 블렌드를 만들어 total을 낸다 (§8의 같은 함수를 여기서도 쓸 수
        있도록 재정의 — 커널을 새로 띄우고 §12만 돌리는 경우가 있다)."""
        pr = {}
        for g in TARGET_COLS:
            cap = CAPACITY_KWH[g]
            w = res["hp"]["w"][g]
            pr[g] = np.clip((1 - w) * res["pred"]["lgb"][g] + w * res["pred_seeds"][g][si], 0, cap)
        return metric(HOLDOUT_ANS, pd.DataFrame(pr))[0]

    # 커널이 죽어 §3/§7의 runs가 없을 수 있다. 그때는 §7에서 실제로 측정한 값을 쓴다
    # (experiments/log.csv의 exp017=A, exp018=A' 및 §8 출력의 시드노이즈).
    REF = {"A":  {"total": 0.645800, "noise": 0.00050},
           "A2": {"total": 0.648721, "noise": 0.00066}}
    ref = {}
    for k in ("A", "A2"):
        if k in runs:
            sd = np.array([_blend_from_seed(runs[k], i) for i in range(len(NN_SEEDS))]).std(ddof=1)
            ref[k] = {"total": runs[k]["blend"]["total"], "noise": sd / np.sqrt(len(NN_SEEDS)),
                      "src": "이번 세션 측정"}
        else:
            ref[k] = {**REF[k], "src": "§7 기록값"}

    sd3 = np.array([_blend_from_seed(rA3, i) for i in range(len(NN_SEEDS))]).std(ddof=1)
    n3 = sd3 / np.sqrt(len(NN_SEEDS))

    print(f"\n{'='*82}")
    print(f"  칸 A   (확정 모델)          {ref['A']['total']:.4f}   시드노이즈 {ref['A']['noise']:.5f}  [{ref['A']['src']}]")
    print(f"  칸 A'  (재튜닝, T_soft .003) {ref['A2']['total']:.4f}   시드노이즈 {ref['A2']['noise']:.5f}  [{ref['A2']['src']}]")
    print(f"  칸 A'' (T_soft {BEST_TS})       {rA3['blend']['total']:.4f}   시드노이즈 {n3:.5f}")
    print("-" * 82)
    for k, label in [("A2", "A'' - A'   격자 확장의 순효과"), ("A", "A'' - A    확정 모델 대비 총 이득")]:
        d = rA3["blend"]["total"] - ref[k]["total"]
        se = float(np.hypot(n3, ref[k]["noise"]))
        print(f"  {label:<28} {d:+.4f}  (SE {se:.5f}, {d / se:+5.1f}배)")
    print(f"{'='*82}")

    d_ext, se_ext = (rA3["blend"]["total"] - ref["A2"]["total"],
                     float(np.hypot(n3, ref["A2"]["noise"])))
    if d_ext / se_ext >= 3:
        print("[판정] ★ 격자 확장 채택 — A''를 제출 구성으로 삼는다.")
    elif d_ext > 0:
        print(f"[판정] 확장 이득은 노이즈 안({d_ext / se_ext:+.1f}배). "
              "A''가 A'보다 나쁘진 않으므로 둘 중 어느 쪽을 제출해도 되지만, "
              "내부검증이 고른 값을 따르는 것이 일관적이다(= A'').")
    else:
        print(f"[판정] 내부검증이 고른 T_soft가 홀드아웃에서는 역행했다({d_ext:+.4f}). "
              "내부검증(2023 하반기 6개월)이 T_soft에 대해 과적합됐을 수 있다 -> A'를 유지한다.")

    print("\n--- 그룹별 블렌드 점수 (A'' ) ---")
    for g in TARGET_COLS:
        print(f"  {g}: {rA3['per_group'][g]:.4f}")
    print(f"\n제출용 설정: T_SOFT={BEST_TS} | "
          + " | ".join(f"{g}: alpha={HP_A3['lgb'][g]['alpha']:.3f} w={HP_A3['w'][g]:.2f}"
                       for g in TARGET_COLS))

=== w 재선택 (T_soft=0.001, 내부검증 2023 하반기) ===
  kpx_group_1: 최적 w=0.60 (내부검증 0.6350)  [A' 때는 0.65]
  kpx_group_2: 최적 w=0.30 (내부검증 0.6614)  [A' 때는 0.35]
  kpx_group_3: 최적 w=1.00 (내부검증 0.5907)  [A' 때는 0.85]

=== 칸 A'': v1 피처 + v1 재튜닝 + T_soft 0.001 ===
  kpx_group_1: 트리 150그루 | MLP 41에폭 x 시드 5개 | w=0.6
  kpx_group_2: 트리 304그루 | MLP 86에폭 x 시드 5개 | w=0.3
  kpx_group_3: 트리 159그루 | MLP 86에폭 x 시드 5개 | w=1.0
  블렌드 0.6492 (1-NMAE 0.8740, FICR 0.4245) | 소요 158초

  칸 A   (확정 모델)          0.6458   시드노이즈 0.00050  [이번 세션 측정]
  칸 A'  (재튜닝, T_soft .003) 0.6487   시드노이즈 0.00066  [이번 세션 측정]
  칸 A'' (T_soft 0.001)       0.6492   시드노이즈 0.00039
----------------------------------------------------------------------------------
  A'' - A'   격자 확장의 순효과        +0.0005  (SE 0.00077,  +0.7배)
  A'' - A    확정 모델 대비 총 이득     +0.0034  (SE 0.00063,  +5.4배)
[판정] 확장 이득은 노이즈 안(+0.7배). A''가 A'보다 나쁘진 않으므로 둘 중 어느 쪽을 제출해도 되지만, 내부검증이 고른 값을 따르는 것이 일관적이다(= A'').

--- 그룹별 블렌드 점수 (A'' ) ---
  kpx_group_1: 0.6612
  kpx_group_2: 0.67

**실행 후 확인할 것**

- **A″ − A′** 가 3배 이상이면 확장이 값을 한 것이다. 0배 근처면 0.003이 사실상 평평한 봉우리라는 뜻이라 그것도 정보다(더 낮출 이유가 사라진다).
- **A″ − A** 가 제출 판단의 숫자다. §11-4의 A′−A(+0.0029, 3.5배)보다 커졌는지 본다.
- **A″가 A′보다 낮게 나오면** 내부검증 과적합 신호다. 이때는 A′를 유지하고, 그 사실을 기록한다(“내부검증 6개월로 `T_soft`를 소수 셋째 자리까지 고르는 건 무리”라는 교훈).
- w가 A′ 때(0.65/0.35/0.85)와 얼마나 달라졌는지 — 많이 흔들리면 w 선택 자체가 불안정하다는 뜻이다.


In [15]:
# ---------- 12-6. 실험 로그 기록 (A''를 잰 경우에만) ----------
# §10과 달리 이 셀은 스스로 필요한 것을 정의한다 -- §10을 다시 돌리면 2x2 행이 중복으로 쌓이기 때문.
if not IMPROVED or "A3" not in runs:
    print("A''를 재지 않았으므로 기록할 것이 없다.")
else:
    def _git_state():
        try:
            h = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip()
            d = subprocess.run(["git", "status", "--porcelain"],
                               capture_output=True, text=True).stdout.strip()
            return f"{h}({'dirty' if d else 'clean'})"
        except Exception:
            return "unknown"

    def _by_group(answer_df, pred_df):
        out = {}
        for g in TARGET_COLS:
            a = answer_df[g].to_numpy(dtype=float)
            f = pred_df[g].to_numpy(dtype=float)
            cap = CAPACITY_KWH[g]
            v = a >= cap * SCORE_THRESHOLD
            a, f = a[v], f[v]
            er = np.abs(f - a) / cap
            price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
            out[g] = {"nmae": float(np.mean(er)), "ficr": float(np.sum(a * price) / np.sum(a * 4.0))}
        return out

    LOG_PATH = EXP_DIR / "log.csv"
    log = pd.read_csv(LOG_PATH, encoding="utf-8-sig")
    next_id = max(int(s[3:]) for s in log["exp_id"] if str(s).startswith("exp")) + 1

    res, hp = runs["A3"], runs["A3"]["hp"]
    bg = _by_group(HOLDOUT_ANS, pd.DataFrame(res["pred"]["blend"]))
    alpha_str = "/".join("{:.2f}".format(hp["lgb"][g]["alpha"]) for g in TARGET_COLS)
    w_str = "/".join("{:.2f}".format(hp["w"][g]) for g in TARGET_COLS)
    row = {"exp_id": f"exp{next_id:03d}", "date": pd.Timestamp.now().strftime("%Y-%m-%d"),
           "git_hash": _git_state(),
           "model": "blend(lightgbm_quantile + mlp_metric_loss)",
           "features": res["name"],
           "total_score": round(res["blend"]["total"], 6),
           "one_minus_nmae": round(res["blend"]["one_minus_nmae"], 6),
           "ficr": round(res["blend"]["ficr"], 6),
           "val_period": "2024-01-01~2024-12-31",
           "fit_seconds": round(res["elapsed"], 1),
           "public_score": "",
           "note": (f"06 §12 T_soft 격자 하방 확장 | 피처 179개 | T_soft={hp['t_soft']} | "
                    f"alpha={alpha_str} | w={w_str} | "
                    f"LGB단독 {res['lgb']['total']:.4f} MLP단독 {res['mlp']['total']:.4f}")}
    for g in TARGET_COLS:
        row[f"nmae_g{g[-1]}"] = round(bg[g]["nmae"], 6)
        row[f"ficr_g{g[-1]}"] = round(bg[g]["ficr"], 6)

    new_df = pd.DataFrame([row])
    unknown = [c for c in new_df.columns if c not in log.columns]
    assert not unknown, f"log.csv에 없는 컬럼을 만들려 합니다: {unknown}"
    pd.concat([log, new_df], ignore_index=True)[log.columns].to_csv(
        LOG_PATH, index=False, encoding="utf-8-sig")
    print(f"1개 행 추가 -> {LOG_PATH}")
    print(new_df[["exp_id", "features", "total_score", "one_minus_nmae", "ficr"]].to_string(index=False))

1개 행 추가 -> d:\공모전\wind_forecast\experiments\log.csv
exp_id             features  total_score  one_minus_nmae     ficr
exp033 A3_v1feat_tsoft0.001     0.649242        0.873992 0.424492


**실행 후 확인할 것**

- `exp022`가 추가돼야 한다(기존 마지막이 exp021).
- **이 셀도 여러 번 돌리면 행이 중복으로 쌓인다.** 한 번만 실행한다.


### 12-7. 결과 정리 — 봉우리는 찾았는데, 홀드아웃에서는 값을 안 했다

**세 줄 요약**: `T_soft` 봉우리는 **0.001에서 격자 안에 갇혔다**(12-1의 기전 예측 적중). 내부검증은 크게 올랐다(+0.0045). **그런데 2024 홀드아웃에서는 +0.0005(0.7배)로 사실상 움직이지 않았다.**

#### (1) 측정은 믿을 수 있다

`T_soft=0.003`을 다시 재서 §5의 내부검증 평균 **0.6196이 소수점 네 자리까지 그대로** 나왔다(차이 −0.0000). 아래 숫자들은 같은 저울에서 잰 값이다.

#### (2) `T_soft` 곡선 — 봉우리를 찾았다

| `T_soft` | 내부검증 평균 | |
|---:|---:|---|
| 0.0005 | 0.6256 | |
| **0.001** | **0.6265** | **← 최적** |
| 0.002 | 0.6203 | |
| 0.003 | 0.6196 | (§5의 선택) |
| 0.004 | 0.6173 | |
| 0.006 | 0.6173 | (확정 모델의 값) |
| 0.012 | 0.6152 | |
| 0.025 | 0.6109 | |

**12-1에서 예측한 기전이 그대로 나타났다.** 0.001에서 정점을 찍고 **0.0005에서는 다시 내려간다**(−0.0009). "T를 낮추면 손실이 산식에 가까워지지만 기울기를 받는 샘플이 밴드 경계 ±T 창 안으로만 좁아진다"는 상충이 실제 데이터에서 확인된 것이다. 이제 봉우리가 양옆으로 갇혔으므로 **이 축은 더 팔 것이 없다.**

곡선 모양도 눈여겨볼 만하다. 0.002~0.025 구간은 완만한데(0.6203 → 0.6109), **0.002 → 0.001에서 갑자기 +0.0062로 꺾인다.** 부드럽지 않다는 것 자체가 뒤에서 볼 불안정성의 조짐이다.

#### (3) 홀드아웃 — 내부검증의 개선이 실현되지 않았다

| 칸 | `T_soft` | w (g1/g2/g3) | **2024 블렌드** | 1−NMAE | FICR | 시드노이즈 |
|---|---:|---|---:|---:|---:|---:|
| A (확정) | 0.006 | 0.80 / 0.50 / 0.90 | 0.6458 | 0.8744 | 0.4172 | 0.00050 |
| A′ | 0.003 | 0.65 / 0.35 / 0.85 | 0.6487 | 0.8744 | 0.4231 | 0.00066 |
| **A″** | **0.001** | **0.60 / 0.30 / 1.00** | **0.6492** | 0.8740 | **0.4245** | 0.00039 |

| 비교 | 값 | SE | 배수 |
|---|---:|---:|---:|
| A″ − A′ (격자 확장의 순효과) | +0.0005 | 0.00077 | **+0.7** |
| A″ − A (확정 모델 대비 총 이득) | +0.0034 | 0.00063 | **+5.4** |

**★ 이번 절의 가장 중요한 관찰 — 실현율이 11%다.**

내부검증에서 블렌드까지 포함한 점수는 A′ 0.6245 → A″ 0.6290으로 **+0.0045** 올랐다. 그런데 홀드아웃에서는 **+0.0005**만 실현됐다.

내부검증 구간은 **2023년 하반기 6개월**이다. `T_soft`를 0.003에서 0.001로 세 배 가까이 좁히는 선택을 그 6개월로 고른 것인데, **그 6개월에만 맞는 값을 고른 것**으로 보인다. 곡선이 0.002 → 0.001에서 계단처럼 튄 것도 같은 이야기다 — 매끄러운 최적화가 아니라 특정 구간에 우연히 잘 맞은 점을 집은 모양이다.

**교훈으로 남긴다: 내부검증 개선폭을 액면 그대로 믿으면 안 된다.** 앞으로 이 하네스로 후보를 고를 때, 내부검증에서 크게 오른 것일수록 홀드아웃에서 확인하기 전에는 기대를 낮춰 잡는다.

#### (4) 그래도 A″가 A보다는 확실히 낫다

`A″ − A = +0.0034(5.4배)`로 **A′(+0.0029/3.5배)보다 오히려 강하다.** 그리고 시드노이즈가 0.00039로 셋 중 가장 작다 — 예측이 더 안정적이다.

다만 **A′와 A″는 서로 구분되지 않는다**(0.7배). 즉 이 두 구성 중 어느 쪽을 골라도 홀드아웃 상으로는 같은 값이다.

패턴은 여전하다 — **1−NMAE는 0.8744 → 0.8740으로 오히려 조금 내려갔는데 FICR이 0.4231 → 0.4245로 올라 총점을 만들었다.** `T_soft`를 낮추는 것의 타깃이 FICR이므로 원인·결과가 맞는다.

#### (5) 부산물 둘 — 다음 실험에 직접 쓰인다

**(a) 그룹별 `T_soft`(권장 순서 5번)의 사전 신호가 나왔다.**

| 그룹 | 그룹별 최적 `T_soft` | 공통 0.001 대비 |
|---|---:|---:|
| g1 | 0.0005 | +0.0000 |
| **g2** | **0.002** | **+0.0017** |
| g3 | 0.001 | +0.0000 |

**g2만 다른 값을 원한다.** 세 그룹 중 가장 잘 맞는 그룹(NMAE 0.1265)이 **더 뭉갠 손실**을 원한다는 것이 흥미롭다. 다만 위 (3)의 실현율 11%를 감안하면 이 +0.0017도 그대로 믿을 수는 없다 — **기대는 낮춰 잡되, 축 자체는 유효하다.**

**(b) g3의 블렌드 가중치가 w=1.00으로 갔다 — LightGBM을 완전히 버렸다.**

격자 상한값이다. g3 예측의 **100%를 MLP가 만든다**는 뜻이고, 두 모델의 오차가 서로 상쇄되는 블렌드의 이점을 g3에서는 포기한 것이다.

이건 구v5 `05_tuning_1` 14-8에서도 똑같이 나왔던 현상이고, 그때 해석은 *"라벨 기간이 짧아 데이터가 적은데, MLP의 매끄러운 함수 근사가 이런 소규모·저신호 상황에서 트리보다 덜 과적합했을 가능성"* 이었다. **g3의 병목이 표본 부족이라는 진단을 가리키는 여섯 번째 경로다.**

그리고 이것이 **권장 순서 3번(g3만 통합 학습)의 기대치를 올린다** — w=1.00이면 **g3 MLP의 개선이 g3 점수에 100% 그대로 반영**되고, g3는 총점의 3분의 1이다.

#### (6) 판정과 다음 행동

- **`T_soft` 축은 종료.** 봉우리가 갇혔고 더 팔 것이 없다.
- **제출 구성**: A′와 A″가 홀드아웃에서 구분되지 않으므로 **둘 다 제출해 리더보드로 직접 가른다.** 제출은 하루 5회 가능하고 마감까지 여유가 있다. 이 비교는 *"내부검증 6개월로 `T_soft`를 잘게 고르는 것이 실제로 되는가"* 에 대한 실측 답을 주고, 그 답은 이후 모든 실험의 판정 기준에 영향을 준다.
- 그다음은 권장 순서 3번(**g3만 통합 학습**). 비교 기준선은 A가 아니라 **리더보드가 더 낫다고 판정한 쪽**(A′ 또는 A″)이다.


## 13. 시간 분할 진단 — 판정 기준으로 쓰려 했으나 기각

**결론 먼저: 여기서 만들려던 "채택 판정 기준"은 기각됐다. 구간별 채점 도구(`segment_scores`)만 진단용으로 남긴다.**

### 왜 시작했나

§12까지 오면서 이 하네스가 **첫 실전 시험에서 순위를 정확히 반대로 매겼다.**

| 구성 | 2024 홀드아웃 | **2025 Public** |
|---|---:|---:|
| A (exp017) | 0.6458 (3위) | **0.638863 (1위)** |
| A′ (exp028) | 0.6487 (2위) | 0.634459 (2위) |
| A″ (exp027) | **0.6492 (1위)** | 0.631730 (3위) |

A″는 §8 판정을 **시드 노이즈의 5.4배**로 통과했는데 리더보드에서 −0.0071로 졌다. §8이 재는 노이즈는 **시드(난수)를 바꿨을 때의 흔들림**인데, 정작 걱정해야 할 것은 **연도가 바뀌었을 때의 흔들림**이었다. 그래서 2024를 12개월·4분기로 쪼개 "이득이 고르게 퍼졌는지"를 보는 판정 기준을 만들려 했다.

### 왜 기각했나 — 음성 대조군에서 무너졌다

두 종류를 시도했고 둘 다 실패했다.

| 시도 | A′(실패) | A″(실패) | **블렌드 도입(성공, Public +0.0160)** | 결과 |
|---|---|---|---|---|
| ① 부호 일관성(양수 구간 비율) | 통과 ❌ | 경고 ✔ | 통과 ✔ | **A′를 놓침** |
| ② 시간 추세(기울기·Kendall τ) | 경고 ✔ | 경고 ✔ | **경고 ✘** | **성공까지 경고** |

②는 실패 두 건을 다 잡아서 유망해 보였지만, **리더보드에서 실제로 성공한 변경(exp016 → exp017, +0.0160)에도 경고를 띄웠다.** 추세 외삽도 −0.0028을 예측했는데 실제는 +0.0160으로 **부호까지 틀렸다.**

즉 **"이득이 2024 안에서 시간에 따라 감소한다"는 것은 실패의 특징이 아니라 이 데이터의 일반 성질**이다. 성공한 변경도 같은 추세를 보인다. 그걸로는 아무것도 가릴 수 없다.

### 왜 문턱을 다시 안 고쳤나

표에서 문턱을 조정하면 3/3을 맞출 수 있다(예: "월별 양수 11/12 이상"). **그렇게 하지 않았다.**

- 지표를 이미 두 번 갈아 끼웠다(부호 → 추세). 세 번째는 데이터를 설명하는 게 아니라 **외우는** 것이다.
- 표본이 **3개**뿐이고, 그중 A′·A″는 LightGBM이 비트 단위로 같아 **사실상 독립이 아니다.**
- §13을 시작할 때 "결과를 보고 지표를 바꾸면 검증이 아니다"라고 적어 뒀다. 그 규칙을 지킨다.

### 그리고 애초에 이 도구가 필요하지 않았다

**제출은 부족한 자원이 아니다** — 마감까지 16일 × 하루 5회 ≈ **80회**인데 남은 후보는 6~8개다. 후보 하나당 열 번씩 제출할 수 있다. 그런데 제출 몇 번을 아끼려고 탐지기를 만들고 검증하는 데 반나절을 썼다.

**가장 정확하고 값싼 판정 기준은 리더보드 그 자체다.**

### 그래서 앞으로 하네스를 쓰는 방식 (원칙)

| 쓴다 | 안 쓴다 |
|---|---|
| 변형 여러 개를 **싸게 비교해 후보를 좁히기** (§12에서 `T_soft` 8개 값을 훑은 것처럼) | **"리더보드에서도 오를까"의 답으로 쓰기** |
| 코드가 망가지지 않았는지 **회귀 확인** | 홀드아웃 이득만 보고 **채택 결정** |

**채택은 제출로 판정한다.** 홀드아웃 이득은 후보를 고르는 참고치일 뿐이다.

---

아래는 남겨 둔 진단 도구다. 구간별로 점수를 쪼개 보는 것 자체는 오류 분석에 쓸모가 있어 보존한다.


In [16]:
# ---------- 13-0. 세 구성의 홀드아웃 예측 확보 (A / A' / A'') ----------
# 커널이 살아 있으면 §7·§12의 결과를 그대로 쓰고, 없으면 그 구성만 다시 학습한다.
NEED = {
    "A":  ("A  v1피처 + 확정설정(exp017)", HP_V1),
    "A2": ("A' v1피처 + v1재튜닝",         None),   # 아래에서 채운다
    "A3": ("A'' A' + T_soft 0.001",       None),
}

if "runs" not in globals():
    runs = {}

# A'·A''의 하이퍼파라미터를 (메모리 -> 파일 -> 재계산) 순으로 확보
if "HP_V1_RETUNED" not in globals():
    HP_V1_RETUNED = json.loads((EXP_DIR / "hp_v1_retuned.json").read_text(encoding="utf-8"))
NEED["A2"] = (NEED["A2"][0], {"lgb": HP_V1_RETUNED["lgb"], "t_soft": 0.003,
                              "w": {"kpx_group_1": 0.65, "kpx_group_2": 0.35, "kpx_group_3": 0.85}})
NEED["A3"] = (NEED["A3"][0], {"lgb": HP_V1_RETUNED["lgb"], "t_soft": 0.001,
                              "w": {"kpx_group_1": 0.60, "kpx_group_2": 0.30, "kpx_group_3": 1.00}})

for key, (label, hp) in NEED.items():
    if key in runs:
        print(f"{key}: 캐시 재사용 (블렌드 {runs[key]['blend']['total']:.4f})")
    else:
        print(f"{key}: 캐시에 없어 다시 학습합니다 — {label}")
        runs[key] = run_config(feat_v1, COLS_V1, hp, label, verbose=False)
        print(f"   -> 블렌드 {runs[key]['blend']['total']:.4f}")

# 홀드아웃 구간의 시각 (구간을 자르는 기준)
HO_DTM = feat_v1.loc[holdout_mask, "forecast_kst_dtm"].reset_index(drop=True)
print(f"\n홀드아웃 {len(HO_DTM):,}행 | {HO_DTM.min()} ~ {HO_DTM.max()}")

A: 캐시 재사용 (블렌드 0.6458)
A2: 캐시 재사용 (블렌드 0.6487)
A3: 캐시 재사용 (블렌드 0.6492)

홀드아웃 8,784행 | 2024-01-01 01:00:00 ~ 2025-01-01 00:00:00


### 13-1. 구간별 채점 (진단용)

구간마다 **대회 공식 산식(`metric()`)을 그대로** 적용한다. 정답도 예측도 같은 행만 잘라 넣으므로 산식은 손대지 않는다.

**판정에 쓰지 않는다.** "어느 계절에 약한가" 같은 오류 분석용이다.


In [ ]:
# ---------- 13-1. 구간별 채점 (진단용) ----------
def segment_scores(res, seg_index, min_rows=200):
    """구간마다 대회 공식 산식을 그대로 적용해 total을 낸다.
    min_rows 미만인 구간은 버린다 -- 표본이 너무 작으면 FICR(비율)이 요동친다."""
    pred_df = pd.DataFrame(res["pred"]["blend"]).reset_index(drop=True)
    out = {}
    for seg in sorted(seg_index.unique()):
        pos = np.flatnonzero((seg_index == seg).to_numpy())
        if len(pos) < min_rows:
            continue
        out[seg] = metric(HOLDOUT_ANS.iloc[pos], pred_df.iloc[pos])[0]
    return pd.Series(out)


# 대회의 1년 구조가 "1/1 01:00 ~ 익년 1/1 00:00"이라, 홀드아웃 마지막 시각(2025-01-01 00:00)은
# 달력상 2025년이지만 실제로는 '2024-12-31의 24시'다. 그대로 자르면 1행짜리 13번째 달이 생긴다.
# 한 시간을 빼서 나누면 정확히 12개월 / 4분기가 된다.
SEG_DTM = HO_DTM - pd.Timedelta(hours=1)
MONTH = SEG_DTM.dt.to_period("M").astype(str)
QUART = "Q" + SEG_DTM.dt.quarter.astype(str)
assert MONTH.nunique() == 12 and QUART.nunique() == 4, (MONTH.nunique(), QUART.nunique())

for scale, seg in [("월", MONTH), ("분기", QUART)]:
    df = pd.DataFrame({k: segment_scores(runs[k], seg) for k in ["A", "A2", "A3"]})
    d = pd.DataFrame({f"{k}-A": df[k] - df["A"] for k in ["A2", "A3"]})
    print(f"\n[{scale}별 홀드아웃 total]  (A = 확정 모델 exp017)")
    print(df.round(4).join(d.round(4)).to_string())